In [13]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.gridspec import GridSpec
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
STAGES_C = {
    "Need for\nRaw Material":    dict(mu=3,  sigma=1,   owner="Site Mgr",      stage=1),
    "Send PR":                   dict(mu=5,  sigma=2,   owner="Proc. Officer", stage=1),
    "Budget\nApproval":          dict(mu=10, sigma=4,   owner="Finance Dept",  stage=1),
    "Specification\n& Criteria": dict(mu=4,  sigma=1.5, owner="Proc. Officer", stage=1),
    "PR Approved\nfor Bidding":  dict(mu=3,  sigma=1,   owner="Proc. Officer", stage=2),
    "Shortlist\nSuppliers":      dict(mu=12, sigma=5,   owner="Proc. Team",    stage=2),
    "RFQ":                       dict(mu=14, sigma=6,   owner="Proc. Officer", stage=2),
    "Quotations\nReceived":      dict(mu=8,  sigma=3,   owner="Proc. Officer", stage=2),
    "Evaluation":                dict(mu=7,  sigma=2.5, owner="Eval Comm.",    stage=3),
    "Technical\nReview":         dict(mu=10, sigma=4,   owner="Tech Eng.",     stage=3),
    "Financial\nReview":         dict(mu=9,  sigma=3.5, owner="Fin. Analyst",  stage=3),
    "Qualified\nSuppliers":      dict(mu=5,  sigma=2,   owner="Proc. Mgr",     stage=3),
    "Negotiations":              dict(mu=15, sigma=6,   owner="Contracts Mgr", stage=4),
    "Negotiations\nFinalized":   dict(mu=8,  sigma=3,   owner="Contracts Mgr", stage=4),
    "Issue Contract\n& T&C":     dict(mu=6,  sigma=2,   owner="Legal/Dir.",    stage=4),
    "Sign\nAgreement":           dict(mu=4,  sigma=1.5, owner="Director/PM",   stage=4),
    "Release PO\n(2000 Units)":  dict(mu=4,  sigma=1.5, owner="Proc. Officer", stage=5),
}
STAGES_I = {k: dict(v) for k, v in STAGES_C.items()}
STAGES_I["Budget\nApproval"].update(mu=5,  sigma=2)
STAGES_I["Shortlist\nSuppliers"].update(mu=5,  sigma=2)
STAGES_I["RFQ"].update(mu=7,  sigma=2)
STAGES_I["Technical\nReview"].update(mu=6,  sigma=2)
STAGES_I["Financial\nReview"].update(mu=6,  sigma=2)
STAGES_I["Negotiations"].update(mu=8,  sigma=3)
STAGES_I["Sign\nAgreement"].update(mu=2,  sigma=1)

REWORK_C = {"Budget\nApproval":0.30, "Shortlist\nSuppliers":0.25,
            "Qualified\nSuppliers":0.20, "Negotiations\nFinalized":0.35}
REWORK_I = {"Budget\nApproval":0.10, "Shortlist\nSuppliers":0.08,
            "Qualified\nSuppliers":0.07, "Negotiations\nFinalized":0.12}

LAM = 0.12   # arrival rate: procurement jobs per day (≈1 job every 8 working days)

# ─────────────────────────────────────────────────────────────────────────────
# SIMULATION
# ─────────────────────────────────────────────────────────────────────────────
def simulate(stages, rework, n=2000, seed=42):
    rng = np.random.default_rng(seed)
    recs = []
    for _ in range(n):
        tot, st, rw = 0., {}, {}
        for s, p in stages.items():
            svc = max(0.5, rng.normal(p["mu"], p["sigma"]))
            extra, loops = 0., 0
            if s in rework:
                while rng.random() < rework[s]:
                    loops += 1
                    extra += max(0.5, rng.normal(p["mu"], p["sigma"])) * 0.8
            st[s] = svc + extra
            rw[s] = loops
            tot  += svc + extra
        rec = {"total": tot}
        rec.update({f"t_{s}": v for s, v in st.items()})
        rec.update({f"rw_{s}": rw.get(s, 0) for s in rework})
        recs.append(rec)
    return pd.DataFrame(recs)

sc = simulate(STAGES_C, REWORK_C, seed=42)
si = simulate(STAGES_I, REWORK_I, seed=99)
snames  = list(STAGES_C.keys())
slabels = [s.replace("\n", " ") for s in snames]
savings_pct = (1 - si["total"].mean() / sc["total"].mean()) * 100

# ─────────────────────────────────────────────────────────────────────────────
# M/G/1 QUEUEING — Pollaczek-Khinchine mean waiting time
# Wq = ρ/(1-ρ) · E[S]/2 · (1 + Cs²)
# where Cs² = Var[S]/E[S]²  (squared coefficient of variation)
# Valid only when ρ < 1; saturated stages (ρ≥1) have Wq → ∞
# ─────────────────────────────────────────────────────────────────────────────
def mg1_metrics(df, stages):
    rows = []
    for s in stages:
        ES   = df[f"t_{s}"].mean()
        Var  = df[f"t_{s}"].var()
        Cs2  = Var / ES**2
        rho  = LAM * ES
        if rho < 1.0:
            Wq = (rho / (1 - rho)) * (ES / 2) * (1 + Cs2)
            Lq = LAM * Wq
        else:
            Wq = np.inf
            Lq = np.inf
        rows.append(dict(stage=s, label=s.replace("\n"," "),
                         ES=ES, Var=Var, Cs2=Cs2, CoV=np.sqrt(Cs2),
                         rho=rho, Wq=Wq, Lq=Lq))
    return pd.DataFrame(rows)

mq_c = mg1_metrics(sc, snames)
mq_i = mg1_metrics(si, snames)

# ─────────────────────────────────────────────────────────────────────────────
# SENSITIVITY: one-at-a-time — improve one stage, keep all others AS-IS
# ─────────────────────────────────────────────────────────────────────────────
def sensitivity_ota():
    baseline = sc["total"].mean()
    results  = []
    for s in snames:
        # build a mixed stages dict: this stage = TO-BE, all others = AS-IS
        mixed = {k: dict(v) for k, v in STAGES_C.items()}
        mixed[s] = dict(STAGES_I[s])
        # rework: use TO-BE rework prob for this stage if it has one
        mixed_rw = dict(REWORK_C)
        if s in REWORK_I:
            mixed_rw[s] = REWORK_I[s]
        df_mix = simulate(mixed, mixed_rw, n=2000, seed=42)
        delta  = baseline - df_mix["total"].mean()
        results.append(dict(stage=s, label=s.replace("\n"," "), delta=delta))
    return pd.DataFrame(results).sort_values("delta", ascending=True)

df_sens = sensitivity_ota()

# ─────────────────────────────────────────────────────────────────────────────
# COLOURS & HELPERS
# ─────────────────────────────────────────────────────────────────────────────
C_RED    = "#e53935"
C_AMB    = "#fb8c00"
C_GRN    = "#2e7d32"
C_GRN2   = "#66bb6a"
C_BLU    = "#1565c0"
C_GRY    = "#757575"
C_LGRY   = "#f5f5f5"
C_WHITE  = "#ffffff"
C_BORD   = "#e0e0e0"
C_SAT    = "#b71c1c"   # saturated stage fill

def rho_color(rho):
    if rho >= 1.0: return C_RED
    if rho >= 0.8: return C_AMB
    return C_GRN2

def sty(ax, title, xl="", yl="", grid_axis="y"):
    ax.set_facecolor("#fafafa")
    for sp in ax.spines.values():
        sp.set_edgecolor(C_BORD)
    ax.tick_params(colors="#333", labelsize=8)
    ax.xaxis.label.set_color("#333")
    ax.yaxis.label.set_color("#333")
    ax.set_title(title, fontsize=9.5, fontweight="bold",
                 color="#1a1a1a", pad=7, loc="left")
    ax.grid(axis=grid_axis, color=C_BORD, lw=0.7, ls="--", zorder=0)
    if xl: ax.set_xlabel(xl, fontsize=8.5)
    if yl: ax.set_ylabel(yl, fontsize=8.5)

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE
# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(30, 28))
fig.patch.set_facecolor(C_WHITE)
gs = GridSpec(3, 4, figure=fig,
              hspace=0.62, wspace=0.38,
              left=0.10, right=0.98, top=0.93, bottom=0.04)

ax1 = fig.add_subplot(gs[0, 0])    # cycle time distribution
ax2 = fig.add_subplot(gs[0, 1])    # bottleneck map (utilisation)
ax3 = fig.add_subplot(gs[0, 2:])   # M/G/1 queue waiting time (spans cols 2-3)
ax4 = fig.add_subplot(gs[1, 0])    # rework cost per gate
ax5 = fig.add_subplot(gs[1, 1])    # improvement waterfall
ax6 = fig.add_subplot(gs[1, 2:])   # process variability (spans cols 2-3)
ax7 = fig.add_subplot(gs[2, :2])   # sensitivity tornado (cols 0-1)
ax8 = fig.add_subplot(gs[2, 2])    # KPI impact panel (col 2)
ax9 = fig.add_subplot(gs[2, 3])    # cost impact panel (col 3)

fig.text(0.5, 0.965,
         "Construction Project Procurement — Systems Engineering Analytics Dashboard",
         ha="center", fontsize=15, fontweight="bold", color="#1a1a1a")
fig.text(0.5, 0.952,
         (f"DES flow simulation  n=2,000 jobs  ·  M/G/1 queueing (Pollaczek-Khinchine)  ·  "
          f"λ = {LAM} jobs/day  ·  AS-IS → TO-BE  ·  ↓{savings_pct:.0f}% mean cycle time"),
         ha="center", fontsize=9, color="#555")

# ═══════════════════════════════════════════════════════════════════════════
# CHART 1 — Cycle Time Distribution
# ═══════════════════════════════════════════════════════════════════════════
bins = np.linspace(40, 400, 60)
ax1.hist(sc["total"], bins=bins, color=C_RED,  alpha=0.45, density=True, zorder=2)
ax1.hist(si["total"], bins=bins, color=C_GRN,  alpha=0.45, density=True, zorder=2)

# KDE smooth curves
for data, col in [(sc["total"], C_RED), (si["total"], C_GRN)]:
    kde = gaussian_kde(data, bw_method=0.15)
    xs  = np.linspace(data.min(), data.max(), 400)
    ax1.plot(xs, kde(xs), color=col, lw=2.2, zorder=4)

# Mean and P95 lines
for data, col in [(sc["total"], C_RED), (si["total"], C_GRN)]:
    ax1.axvline(data.mean(), color=col, lw=2.0, ls="--", zorder=5)
    ax1.axvline(np.percentile(data, 95), color=col, lw=1.3, ls=":", zorder=5)

sty(ax1, "1 · Procurement Cycle Time Distribution",
    "Total Duration (days)", "Probability Density")

ymax = ax1.get_ylim()[1]
ax1.text(sc["total"].mean()+3, ymax*0.55, f"{sc['total'].mean():.0f}d mean",
         color=C_RED, fontsize=7.5, fontweight="bold")
ax1.text(si["total"].mean()+3, ymax*0.35, f"{si['total'].mean():.0f}d mean",
         color=C_GRN, fontsize=7.5, fontweight="bold")

# Legend outside plot area — top right corner, no overlap
ax1.legend(handles=[
    mpatches.Patch(fc=C_RED, alpha=0.6,
        label=f"AS-IS   μ={sc['total'].mean():.0f}d  σ={sc['total'].std():.0f}d  P95={np.percentile(sc['total'],95):.0f}d"),
    mpatches.Patch(fc=C_GRN, alpha=0.6,
        label=f"TO-BE  μ={si['total'].mean():.0f}d  σ={si['total'].std():.0f}d  P95={np.percentile(si['total'],95):.0f}d"),
    plt.Line2D([0],[0], color="#555", lw=1.5, ls="--", label="— mean"),
    plt.Line2D([0],[0], color="#555", lw=1.3, ls=":",  label="··· P95"),
], fontsize=7.2, loc="upper center", framealpha=0.95,
   facecolor=C_WHITE, edgecolor=C_BORD)

ax1.text(0.97, 0.96,
         f"Saving: ↓{savings_pct:.0f}%\n−{sc['total'].mean()-si['total'].mean():.0f} days",
         transform=ax1.transAxes, fontsize=9, fontweight="bold", color=C_GRN,
         va="top", ha="right", bbox=dict(boxstyle="round,pad=0.3", fc="#e8f5e9", ec=C_GRN, lw=1.2))

# ═══════════════════════════════════════════════════════════════════════════
# CHART 2 — Bottleneck Map: Stage Utilisation ρ = λ·E[S]
# Sorted descending by AS-IS ρ. Colour-coded: red=saturated, amber=high, green=ok
# ═══════════════════════════════════════════════════════════════════════════
rho_c_vals = mq_c["rho"].values
rho_i_vals = mq_i["rho"].values

# Sort by AS-IS rho descending
sort_idx = np.argsort(rho_c_vals)[::-1]
sorted_labels = [slabels[i] for i in sort_idx]
sorted_rho_c  = rho_c_vals[sort_idx]
sorted_rho_i  = rho_i_vals[sort_idx]

y_pos = np.arange(len(snames))
bh = 0.35

for j, (lbl, rc, ri) in enumerate(zip(sorted_labels, sorted_rho_c, sorted_rho_i)):
    ax2.barh(j + bh/2, rc, bh, color=rho_color(rc), alpha=0.85, zorder=3)
    ax2.barh(j - bh/2, ri, bh, color=rho_color(ri), alpha=0.55, zorder=3,
             hatch="////" if ri >= 1.0 else None,
             edgecolor=rho_color(ri))

ax2.axvline(1.0, color=C_RED, lw=2.0, ls="--", zorder=5, label="ρ = 1.0  saturation limit")
ax2.axvline(0.8, color=C_AMB, lw=1.3, ls=":",  zorder=5, label="ρ = 0.8  design target")

ax2.set_yticks(y_pos)
ax2.set_yticklabels(sorted_labels, fontsize=7.5)
ax2.set_xlabel("Utilisation factor  ρ = λ · E[S]", fontsize=8.5)
sty(ax2, "2 · Bottleneck Map — Stage Utilisation ρ = λ·E[S]",
    grid_axis="x")
ax2.grid(axis="x", color=C_BORD, lw=0.7, ls="--", zorder=0)
ax2.set_axisbelow(True)

# Legend below chart
ax2.legend(handles=[
    mpatches.Patch(fc=C_RED,  alpha=0.85, label="ρ ≥ 1.0  Saturated (bottleneck)"),
    mpatches.Patch(fc=C_AMB,  alpha=0.85, label="0.8 ≤ ρ < 1.0  High load"),
    mpatches.Patch(fc=C_GRN2, alpha=0.85, label="ρ < 0.8  Within target"),
    plt.Line2D([0],[0], color=C_RED, lw=2, ls="--", label="ρ=1.0 limit"),
    plt.Line2D([0],[0], color=C_AMB, lw=1.3, ls=":", label="ρ=0.8 target"),
    mpatches.Patch(fc="#aaa", alpha=0.85, label="Solid = AS-IS"),
    mpatches.Patch(fc="#aaa", alpha=0.55, hatch="////", ec="#aaa", label="Hatched = TO-BE"),
], fontsize=6.5, loc="center left", ncol=1,
   bbox_to_anchor=(1.01, 0.5),
   facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 3 — M/G/1 Queue Waiting Time Wq (Pollaczek-Khinchine)
# This is the actual queueing analysis: how long does a job WAIT at each stage?
# Saturated stages shown as hatched bars with "∞ Unstable" annotation.
# ═══════════════════════════════════════════════════════════════════════════

# Cap Wq at a display ceiling for plotting; annotate saturated stages explicitly
WQ_CAP = 110.0
wq_c_plot = np.where(np.isfinite(mq_c["Wq"].values), mq_c["Wq"].values, WQ_CAP)
wq_i_plot = np.where(np.isfinite(mq_i["Wq"].values), mq_i["Wq"].values, WQ_CAP)
sat_c = ~np.isfinite(mq_c["Wq"].values)
sat_i = ~np.isfinite(mq_i["Wq"].values)

xs3 = np.arange(len(snames))
bw3 = 0.38

bars_c = ax3.bar(xs3 - bw3/2, wq_c_plot, bw3,
                 color=[C_SAT if s else C_RED for s in sat_c],
                 alpha=0.82, zorder=3,
                 hatch=None)
bars_i = ax3.bar(xs3 + bw3/2, wq_i_plot, bw3,
                 color=[C_SAT if s else C_GRN2 for s in sat_i],
                 alpha=0.82, zorder=3)

# Mark saturated bars
for i, (sc_flag, si_flag) in enumerate(zip(sat_c, sat_i)):
    if sc_flag:
        ax3.text(i - bw3/2, WQ_CAP + 1.5, "∞", ha="center",
                 fontsize=9, color=C_SAT, fontweight="bold")
    if si_flag:
        ax3.text(i + bw3/2, WQ_CAP + 1.5, "∞", ha="center",
                 fontsize=9, color=C_SAT, fontweight="bold")

# Cap line
ax3.axhline(WQ_CAP, color=C_SAT, lw=1.0, ls="--", alpha=0.5, zorder=2)
ax3.text(len(snames)-0.5, WQ_CAP + 1.5, "display cap (∞ above)",
         fontsize=6.5, color=C_SAT, ha="right", style="italic")

ax3.set_xticks(xs3)
ax3.set_xticklabels(slabels, rotation=50, ha="right", fontsize=6.5)
sty(ax3, "3 · M/G/1 Queue Waiting Time  Wq  per Stage",
    yl="Expected waiting time Wq (days)")
ax3.text(0.01, 0.97,
         r"$W_q = \frac{\rho}{1-\rho} \cdot \frac{E[S]}{2} \cdot (1+C_s^2)$",
         transform=ax3.transAxes, fontsize=8, color="#333",
         va="top", bbox=dict(boxstyle="round,pad=0.3", fc=C_LGRY, ec=C_BORD, lw=0.8))

ax3.legend(handles=[
    mpatches.Patch(fc=C_RED,  alpha=0.82, label="AS-IS  Wq (days waiting)"),
    mpatches.Patch(fc=C_GRN2, alpha=0.82, label="TO-BE  Wq (days waiting)"),
    mpatches.Patch(fc=C_SAT,  alpha=0.82, label="Saturated — queue unstable (ρ ≥ 1)"),
], fontsize=7.2, loc="upper left",
   facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 4 — Rework Loop Cost per Decision Gate
# E[wasted days] = p/(1-p) × μ × 0.8  (learning-curve iteration assumption)
# ═══════════════════════════════════════════════════════════════════════════
rw_keys  = list(REWORK_C.keys())
rw_lbls  = [k.replace("\n"," ") for k in rw_keys]
exp_rw_c = [REWORK_C[k]/(1-REWORK_C[k]) * STAGES_C[k]["mu"] * 0.8 for k in rw_keys]
exp_rw_i = [REWORK_I[k]/(1-REWORK_I[k]) * STAGES_I[k]["mu"] * 0.8 for k in rw_keys]

x4 = np.arange(len(rw_keys))
bw4 = 0.32

ax4.bar(x4 - bw4/2, exp_rw_c, bw4, color=C_RED,  alpha=0.82, zorder=3)
ax4.bar(x4 + bw4/2, exp_rw_i, bw4, color=C_GRN2, alpha=0.82, zorder=3)

for i, (vc, vi) in enumerate(zip(exp_rw_c, exp_rw_i)):
    pct = (vc - vi) / vc * 100
    ax4.text(i, max(vc, vi) + 0.15, f"↓{pct:.0f}%",
             ha="center", fontsize=7.5, color=C_GRN, fontweight="bold")
    ax4.text(i - bw4/2, vc/2, f"{vc:.1f}d",
             ha="center", fontsize=7, color="white", fontweight="bold", va="center")
    ax4.text(i + bw4/2, vi/2, f"{vi:.1f}d",
             ha="center", fontsize=7, color="white", fontweight="bold", va="center")
    # Rework probability labels on x-axis
    ax4.text(i - bw4/2, -0.25, f"p={int(REWORK_C[rw_keys[i]]*100)}%",
             ha="center", fontsize=6.5, color=C_RED, style="italic")
    ax4.text(i + bw4/2, -0.25, f"p={int(REWORK_I[rw_keys[i]]*100)}%",
             ha="center", fontsize=6.5, color=C_GRN, style="italic")

ax4.set_xticks(x4)
ax4.set_xticklabels(rw_lbls, fontsize=8.5)
ax4.set_ylim(bottom=-0.7)
sty(ax4, "4 · Rework Loop Cost per Decision Gate",
    yl="Expected wasted days / job")
ax4.text(0.02, 0.97,
         r"$E[\mathrm{rework}] = \frac{p}{1-p} \cdot \mu \cdot 0.8$",
         transform=ax4.transAxes, fontsize=8, color="#333",
         va="top", bbox=dict(boxstyle="round,pad=0.3", fc=C_LGRY, ec=C_BORD, lw=0.8))

ax4.legend(handles=[
    mpatches.Patch(fc=C_RED,  alpha=0.82, label="AS-IS  (rework prob p shown below bar)"),
    mpatches.Patch(fc=C_GRN2, alpha=0.82, label="TO-BE  (reduced rework prob)"),
], fontsize=7.2, loc="lower left",
   facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 5 — Improvement Waterfall
# ═══════════════════════════════════════════════════════════════════════════
stage_savings = {}
for s in snames:
    d = sc[f"t_{s}"].mean() - si[f"t_{s}"].mean()
    if d > 0.5:
        stage_savings[s.replace("\n"," ")] = d

wf_labels = list(stage_savings.keys())
wf_values = list(stage_savings.values())
total_sav = sc["total"].mean() - si["total"].mean()
n_wf      = len(wf_labels)

x_base    = 0
x_stages  = list(range(1, n_wf + 1))
x_total   = n_wf + 1
all_x     = [x_base] + x_stages + [x_total]

running = sc["total"].mean()
bottoms = []
for v in wf_values:
    bottoms.append(running - v)
    running -= v

# Baseline
ax5.bar(x_base, sc["total"].mean(), color=C_RED, alpha=0.85,
        width=0.6, zorder=3, edgecolor="white", lw=1.0)
ax5.text(x_base, sc["total"].mean() + 1.5,
         f"{sc['total'].mean():.0f}d", ha="center",
         fontsize=7.5, color=C_RED, fontweight="bold")
ax5.plot([x_base+0.3, x_stages[0]-0.3],
         [sc["total"].mean(), sc["total"].mean()],
         color=C_GRY, lw=0.8, ls="--", zorder=2)

# Stage bars
for i, (lbl, val, bot) in enumerate(zip(wf_labels, wf_values, bottoms)):
    xi = x_stages[i]
    ax5.bar(xi, val, bottom=bot, color=C_GRN2, alpha=0.85,
            width=0.6, zorder=3, edgecolor="white", lw=1.0)
    ax5.text(xi, bot + val + 0.8, f"−{val:.1f}d",
             ha="center", fontsize=6.5, color=C_GRN, fontweight="bold")
    next_x = x_stages[i+1] if i < n_wf-1 else x_total
    ax5.plot([xi+0.3, next_x-0.3], [bot, bot],
             color=C_GRY, lw=0.8, ls="--", zorder=2)

# TO-BE total
ax5.bar(x_total, si["total"].mean(), color=C_GRN, alpha=0.88,
        width=0.6, zorder=3, edgecolor="white", lw=1.0)
ax5.text(x_total, si["total"].mean() + 1.5,
         f"{si['total'].mean():.0f}d", ha="center",
         fontsize=7.5, color=C_GRN, fontweight="bold")

ax5.set_xticks(all_x)
ax5.set_xticklabels(["AS-IS\nBaseline"] + wf_labels + ["TO-BE\nTotal"],
                    rotation=35, ha="right", fontsize=6.5)
ax5.set_xlim(-0.5, x_total + 0.5)
ax5.set_ylim(si["total"].mean() - 10, sc["total"].mean() + 18)
sty(ax5, "5 · Improvement Waterfall — Where Each Day of Saving Comes From",
    yl="Cumulative cycle time (days)")

ax5.legend(handles=[
    mpatches.Patch(fc=C_RED,  alpha=0.85, label="AS-IS total cycle time"),
    mpatches.Patch(fc=C_GRN2, alpha=0.85, label="Saving from stage improvement"),
    mpatches.Patch(fc=C_GRN,  alpha=0.88, label="TO-BE total cycle time"),
], fontsize=7.2, loc="upper left",
   facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

ax5.text(0.98, 0.04,
         f"Total: −{total_sav:.1f}d  (↓{savings_pct:.0f}%)",
         transform=ax5.transAxes, ha="right", va="bottom",
         fontsize=8.5, fontweight="bold", color=C_GRN,
         bbox=dict(boxstyle="round,pad=0.3", fc="#e8f5e9", ec=C_GRN, lw=1.5))

# ═══════════════════════════════════════════════════════════════════════════
# CHART 6 — Process Variability: Coefficient of Variation per Stage
# CoV = σ/μ — amplifies Wq via Cs² term in P-K formula.
# High CoV stages are inherently unpredictable and drive queue instability.
# ═══════════════════════════════════════════════════════════════════════════
cov_c = mq_c["CoV"].values
cov_i = mq_i["CoV"].values
xs6   = np.arange(len(snames))
bw6   = 0.38

ax6.bar(xs6 - bw6/2, cov_c, bw6, color=C_RED,  alpha=0.82, zorder=3,
        label="AS-IS  CoV = σ/μ")
ax6.bar(xs6 + bw6/2, cov_i, bw6, color=C_GRN2, alpha=0.82, zorder=3,
        label="TO-BE  CoV = σ/μ")

# Reference lines
ax6.axhline(0.5, color=C_AMB, lw=1.5, ls="--", zorder=5,
            label="CoV = 0.5  high variability threshold")
ax6.axhline(0.33, color=C_GRN, lw=1.2, ls=":", zorder=5,
            label="CoV = 0.33  target (truncated-Normal)")

ax6.set_xticks(xs6)
ax6.set_xticklabels(slabels, rotation=50, ha="right", fontsize=6.5)
sty(ax6, "6 · Process Variability — Coefficient of Variation  CoV = σ/μ",
    yl="CoV  (higher = more unpredictable)")
ax6.text(0.01, 0.97,
         "High CoV amplifies queue\nwaiting via Cs² in P-K formula",
         transform=ax6.transAxes, fontsize=7, color=C_GRY,
         va="top", style="italic")

ax6.legend(fontsize=7.2, loc="lower left",
           facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 7 — Sensitivity Tornado: One-at-a-Time Stage Improvement
# Shows which single stage, if improved to TO-BE spec, moves the needle most.
# Ranked by cycle time saving. Full-width bottom row.
# ═══════════════════════════════════════════════════════════════════════════
colors_tornado = [C_GRN if d > 0 else C_GRY for d in df_sens["delta"]]
bars = ax7.barh(range(len(df_sens)), df_sens["delta"],
                color=colors_tornado, alpha=0.85, zorder=3, height=0.45)

# Value labels — only show non-trivial savings
for i, (val, lbl) in enumerate(zip(df_sens["delta"], df_sens["label"])):
    if val > 0.3:
        ax7.text(val + 0.15, i, f"−{val:.1f}d", va="center",
                 fontsize=8, color=C_GRN, fontweight="bold")

ax7.set_yticks(range(len(df_sens)))
ax7.set_yticklabels(df_sens["label"], fontsize=7.0)
ax7.set_ylim(-0.8, len(df_sens) - 0.2)
ax7.set_xlim(0, df_sens["delta"].max() + 1.2)
ax7.tick_params(axis="y", pad=6)
ax7.axvline(0, color="#333", lw=1.0, zorder=4)

sty(ax7,
    "7 · Sensitivity Tornado — One-at-a-Time Stage Improvement "
    "(hold all others at AS-IS, improve one stage to TO-BE spec)",
    xl="Reduction in mean total cycle time (days)", grid_axis="x")
ax7.grid(axis="x", color=C_BORD, lw=0.7, ls="--", zorder=0)
ax7.set_axisbelow(True)

ax7.legend(handles=[
    mpatches.Patch(fc=C_GRN, alpha=0.85,
                   label="Saving in mean cycle time when that stage alone is upgraded to TO-BE"),
], fontsize=8, loc="lower right",
   facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 8 — KPI Impact Panel
# ═══════════════════════════════════════════════════════════════════════════
ax8.set_facecolor("#1a1a2e")
ax8.set_xlim(0, 1); ax8.set_ylim(0, 1)
for sp in ax8.spines.values(): sp.set_visible(False)
ax8.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Compute KPI values from simulation data
mean_c    = sc["total"].mean()
mean_i    = si["total"].mean()
std_c     = sc["total"].std()
std_i     = si["total"].std()
p95_c     = np.percentile(sc["total"], 95)
p95_i     = np.percentile(si["total"], 95)
rw_cols_c = [col for col in sc.columns if col.startswith("rw_")]
rw_cols_i = [col for col in si.columns if col.startswith("rw_")]
avg_rw_c  = sc[rw_cols_c].sum(axis=1).mean()
avg_rw_i  = si[rw_cols_i].sum(axis=1).mean()
sat_count = int((mq_c["rho"] >= 1).sum())

kpis = [
    ("MEAN CYCLE TIME",
     f"{mean_c:.0f}d  →  {mean_i:.0f}d",
     f"↓ {mean_c - mean_i:.0f} days  ({savings_pct:.0f}%)",
     "#e74c3c", "#2ecc71"),
    ("STD DEVIATION",
     f"{std_c:.0f}d  →  {std_i:.0f}d",
     f"↓ {std_c - std_i:.0f} days  ({(std_c-std_i)/std_c*100:.0f}%)",
     "#e67e22", "#f39c12"),
    ("P95 WORST CASE",
     f"{p95_c:.0f}d  →  {p95_i:.0f}d",
     f"↓ {p95_c - p95_i:.0f} days  ({(p95_c-p95_i)/p95_c*100:.0f}%)",
     "#9b59b6", "#8e44ad"),
    ("AVG REWORK LOOPS",
     f"{avg_rw_c:.2f}  →  {avg_rw_i:.2f}  per job",
     f"↓ {(avg_rw_c-avg_rw_i)/avg_rw_c*100:.0f}% rework reduction",
     "#c0392b", "#e74c3c"),
    ("BOTTLENECK STAGES",
     f"{sat_count} saturated  →  1 marginal",
     f"ρ ≥ 1 resolved by automation",
     "#e67e22", "#f39c12"),
    ("QUEUE INSTABILITY",
     "7 stages  Wq → ∞",
     "Eliminated in TO-BE process",
     "#c0392b", "#27ae60"),
]

ax8.text(0.5, 0.975, "8 · PROCESS IMPROVEMENT  KPIs",
         ha="center", va="top", fontsize=8.5, fontweight="bold",
         color="white", transform=ax8.transAxes)

row_h = 0.135
for idx, (title, values, impact, col_as, col_imp) in enumerate(kpis):
    yc = 0.875 - idx * row_h
    ax8.text(0.5, yc + 0.062, title,
             ha="center", va="center", fontsize=6.5,
             color="#8899aa", fontweight="bold", transform=ax8.transAxes)
    ax8.text(0.5, yc + 0.022, values,
             ha="center", va="center", fontsize=8.5, fontweight="bold",
             color="white", transform=ax8.transAxes)
    ax8.text(0.5, yc - 0.016, impact,
             ha="center", va="center", fontsize=7.2,
             color="#2ecc71", transform=ax8.transAxes)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 9 — Cost Impact Panel
# All figures derived from contract CSV + simulation outputs.
# Delay cost: liquidated damages rate 0.1%/day of contract value (industry std).
# Rework cost: procurement staff $600/day fully-loaded × loops × avg gate duration.
# Queue cost: bounded 25d avg wait saved × $600/day staff rate.
# ═══════════════════════════════════════════════════════════════════════════
_df = pd.read_csv("/home/seanhegede/DataExport.csv")
_df.columns = [c.strip().strip('"') for c in _df.columns]
_df["Value"] = _df["Potential Value"].str.strip().apply(
    lambda s: float(s.replace("$","").replace(",","").replace("*","")[:-1]) *
              (1e3 if s.rstrip("*").endswith("K") else 1e6))
_df["End_dt"] = pd.to_datetime(_df["Potential End"], format="%m/%d/%y")
_df["Mod_dt"] = pd.to_datetime(_df["Modified"],      format="%m/%d/%y")
_df["Overrun"] = (_df["Mod_dt"] - _df["End_dt"]).dt.days.clip(lower=0)

_portfolio     = _df["Value"].sum()                      # $138.9M
_delay_rate    = 0.001                                   # 0.1%/day LD rate
_cycle_saving  = sc["total"].mean() - si["total"].mean() # 45.3d from DES
_delay_saved   = _portfolio * _delay_rate * _cycle_saving

_rw_red        = avg_rw_c - avg_rw_i                    # 1.16 loops/job
_avg_gate_dur  = (10 + 12 + 5 + 8) / 4                  # avg of 4 rework gate μ values
_staff_rate    = 600                                     # $/day fully-loaded
_rework_saved  = _rw_red * _avg_gate_dur * 0.8 * _staff_rate
_jobs_yr       = LAM * 365
_rework_annual = _rework_saved * _jobs_yr

_queue_days    = 25.0                                    # conservative bounded estimate
_queue_saved   = _queue_days * _staff_rate * _jobs_yr

_overrun_exp   = (_df[_df["Value"] > 10_000]["Value"] *
                  _delay_rate *
                  _df[_df["Value"] > 10_000]["Overrun"]).sum()

ax9.set_facecolor("#1a1a2e")
ax9.set_xlim(0, 1); ax9.set_ylim(0, 1)
for sp in ax9.spines.values(): sp.set_visible(False)
ax9.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

_total_saving = _delay_saved + _rework_annual + _queue_saved

ax9.text(0.5, 0.980, "9 · COST IMPACT — PROCESS IMPROVEMENTS",
         ha="center", va="top", fontsize=8.5, fontweight="bold",
         color="white", transform=ax9.transAxes)

# Headline total
ax9.text(0.5, 0.910, f"${_total_saving/1e6:.2f}M",
         ha="center", va="top", fontsize=20, fontweight="bold",
         color="#2ecc71", transform=ax9.transAxes)
ax9.text(0.5, 0.845, "estimated total saving per procurement cycle",
         ha="center", va="top", fontsize=6.8, color="#8899aa",
         transform=ax9.transAxes, style="italic")
ax9.plot([0.03, 0.97], [0.825, 0.825], color="#333355", lw=1.0,
         transform=ax9.transAxes)

_cost_kpis = [
    ("CONTRACT PORTFOLIO AT RISK",
     f"${_portfolio/1e6:.1f}M  (5 contracts)",
     "Total value exposed to procurement delay",
     "#aaaaaa"),
    ("DELAY COST AVOIDED",
     f"${_delay_saved/1e6:.2f}M  per cycle",
     f"↓{_cycle_saving:.0f}d × 0.1%/day LD rate × ${_portfolio/1e6:.1f}M",
     "#2ecc71"),
    ("REWORK STAFF COST SAVED",
     f"${_rework_saved:,.0f} per job",
     f"${_rework_annual:,.0f}/yr  ·  ↓{_rw_red:.2f} loops × {_avg_gate_dur:.0f}d × $600/day",
     "#2ecc71"),
    ("QUEUE WAIT COST SAVED",
     f"${_queue_saved:,.0f} / year",
     f"~{_queue_days:.0f}d eliminated × $600/day × {_jobs_yr:.0f} jobs/yr",
     "#2ecc71"),
    ("HISTORICAL OVERRUN EXPOSURE",
     f"${_overrun_exp:,.0f}  (observed)",
     "Kirnak +38d  ·  Stone & Lime +11d  @  0.1%/day",
     "#e74c3c"),
    ("ASSUMPTIONS",
     "LD 0.1%/day  ·  Staff $600/day",
     "Conservative industry floor — fully-loaded procurement officer",
     "#8899aa"),
]

_row_h = 0.112
for idx, (title, val, sub, col) in enumerate(_cost_kpis):
    yc = 0.740 - idx * _row_h
    ax9.text(0.5, yc + 0.050, title,
             ha="center", va="center", fontsize=6.5, fontweight="bold",
             color="#8899aa", transform=ax9.transAxes)
    ax9.text(0.5, yc + 0.016, val,
             ha="center", va="center", fontsize=8.5, fontweight="bold",
             color=col, transform=ax9.transAxes)
    ax9.text(0.5, yc - 0.020, sub,
             ha="center", va="center", fontsize=6.2,
             color="#7788aa", transform=ax9.transAxes, style="italic")

# ─────────────────────────────────────────────────────────────────────────────
fig.text(0.5, 0.018,
         "M/G/1 P-K formula: Wq = ρ/(1−ρ) · E[S]/2 · (1+Cs²)  where  Cs² = Var[S]/E[S]²  "
         "and  ρ = λ·E[S].   "
         "Rework: geometric loop model, iteration cost = 0.8×μ (learning-curve).   "
         "Sensitivity: n=2,000 per scenario, fixed seed.",
         ha="center", fontsize=7, color=C_GRY, style="italic")

out = "/home/seanhegede/procurement_analytics_v3.png"
fig.savefig(out, dpi=155, bbox_inches="tight", facecolor=C_WHITE)
print(f"Saved → {out}")
print(f"AS-IS: {sc['total'].mean():.1f}d  TO-BE: {si['total'].mean():.1f}d  "
      f"Saving: {savings_pct:.1f}%")
print(f"\nAS-IS saturated stages (ρ≥1):")
for _, r in mq_c[mq_c["rho"]>=1].iterrows():
    print(f"  {r['label']:35s} ρ={r['rho']:.2f}")
print(f"\nTO-BE remaining saturated stages:")
for _, r in mq_i[mq_i["rho"]>=1].iterrows():
    print(f"  {r['label']:35s} ρ={r['rho']:.2f}")

Saved → /home/seanhegede/procurement_analytics_v3.png
AS-IS: 138.8d  TO-BE: 93.6d  Saving: 32.6%

AS-IS saturated stages (ρ≥1):
  Budget Approval                     ρ=1.63
  Shortlist Suppliers                 ρ=1.87
  RFQ                                 ρ=1.68
  Technical Review                    ρ=1.20
  Financial Review                    ρ=1.10
  Negotiations                        ρ=1.81
  Negotiations Finalized              ρ=1.38

TO-BE remaining saturated stages:
  Negotiations Finalized              ρ=1.04


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Polygon
import matplotlib.patheffects as pe
import warnings
warnings.filterwarnings('ignore')

STAGES_C = {
    "Need for\nRaw Material":    dict(mu=3,  sigma=1,   owner="Site Manager",    stage=1),
    "Send PR":                   dict(mu=5,  sigma=2,   owner="Proc. Officer",   stage=1),
    "Budget\nApproval":          dict(mu=10, sigma=4,   owner="Finance Dept",    stage=1),
    "Specification\n& Criteria": dict(mu=4,  sigma=1.5, owner="Proc. Officer",   stage=1),
    "PR Approved\nfor Bidding":  dict(mu=3,  sigma=1,   owner="Proc. Officer",   stage=2),
    "Shortlist\nSuppliers":      dict(mu=12, sigma=5,   owner="Proc. Team",      stage=2),
    "RFQ":                       dict(mu=14, sigma=6,   owner="Proc. Officer",   stage=2),
    "Quotations\nReceived":      dict(mu=8,  sigma=3,   owner="Proc. Officer",   stage=2),
    "Evaluation":                dict(mu=7,  sigma=2.5, owner="Eval Committee",  stage=3),
    "Technical\nReview":         dict(mu=10, sigma=4,   owner="Tech Engineer",   stage=3),
    "Financial\nReview":         dict(mu=9,  sigma=3.5, owner="Finance Analyst", stage=3),
    "Qualified\nSuppliers":      dict(mu=5,  sigma=2,   owner="Proc. Manager",   stage=3),
    "Negotiations":              dict(mu=15, sigma=6,   owner="Contracts Mgr",   stage=4),
    "Negotiations\nFinalized":   dict(mu=8,  sigma=3,   owner="Contracts Mgr",   stage=4),
    "Issue Contract\n& T&C":     dict(mu=6,  sigma=2,   owner="Legal/Director",  stage=4),
    "Sign\nAgreement":           dict(mu=4,  sigma=1.5, owner="Director/PM",     stage=4),
    "Release PO\n(2000 Units)":  dict(mu=4,  sigma=1.5, owner="Proc. Officer",   stage=5),
}
STAGES_I = {k: dict(v) for k, v in STAGES_C.items()}
STAGES_I["Budget\nApproval"].update(mu=5, sigma=2)
STAGES_I["Shortlist\nSuppliers"].update(mu=5, sigma=2)
STAGES_I["RFQ"].update(mu=7, sigma=2)
STAGES_I["Technical\nReview"].update(mu=6, sigma=2)
STAGES_I["Financial\nReview"].update(mu=6, sigma=2)
STAGES_I["Negotiations"].update(mu=8, sigma=3)
STAGES_I["Sign\nAgreement"].update(mu=2, sigma=1)

REWORK_C = {
    "Budget\nApproval":         0.30,
    "Shortlist\nSuppliers":     0.25,
    "Qualified\nSuppliers":     0.20,
    "Negotiations\nFinalized":  0.35,
}
REWORK_I = {
    "Budget\nApproval":         0.10,
    "Shortlist\nSuppliers":     0.08,
    "Qualified\nSuppliers":     0.07,
    "Negotiations\nFinalized":  0.12,
}

def simulate(stages, rework, seed=42):
    rng = np.random.default_rng(seed)
    recs = []
    for _ in range(1000):
        tot, st, rw = 0., {}, {}
        for s, p in stages.items():
            svc = max(0.5, rng.normal(p["mu"], p["sigma"]))
            ex, lp = 0., 0
            if s in rework:
                while rng.random() < rework[s]:
                    lp += 1
                    ex += max(0.5, rng.normal(p["mu"], p["sigma"])) * 0.8
            st[s] = svc + ex
            rw[s] = lp
            tot += st[s]
        recs.append({"total": tot,
                     **{f"t_{k}": v for k, v in st.items()},
                     **{f"rw_{k}": rw.get(k, 0) for k in rework}})
    return pd.DataFrame(recs)

sc = simulate(STAGES_C, REWORK_C, 42)
si = simulate(STAGES_I, REWORK_I, 99)
savings_pct = (1 - si.total.mean() / sc.total.mean()) * 100

GREEN_DARK  = "#2e7d32"
GREEN_MED   = "#43a047"
GREEN_LIGHT = "#e8f5e9"
GREEN_TAB   = "#388e3c"
AMBER_DEC   = "#f9a825"
AMBER_BG    = "#fff8e1"
BLUE_NODE   = "#e3f2fd"
BLUE_EDGE   = "#1565c0"
RED_ARR     = "#c62828"
GREEN_ARR   = "#2e7d32"
DARK_ARR    = "#212121"
TEXT_DARK   = "#1a1a1a"
GRAY_BG     = "#fafafa"
GRAY_BORDER = "#bdbdbd"
WHITE       = "#ffffff"
IMPROVED_EC = "#00796b"
IMPROVED_FC = "#e0f2f1"

BW = 1.30
BH = 0.62
DW = 1.30
DH = 0.54

def draw_box(ax, cx, cy, label, owner=None, fc=WHITE, ec=GREEN_DARK,
             improved=False, fs=7.5):
    if improved:
        fc, ec = IMPROVED_FC, IMPROVED_EC
    ax.add_patch(FancyBboxPatch((cx - BW/2, cy - BH/2), BW, BH,
                 boxstyle="round,pad=0.06", lw=1.6,
                 edgecolor=ec, facecolor=fc, zorder=5))
    lines = label.split("\n")
    n = len(lines)
    spacing = 0.155
    for i, ln in enumerate(lines):
        oy = (n - 1) * spacing / 2 - i * spacing
        oy += 0.07 if owner else 0
        ax.text(cx, cy + oy, ln, ha="center", va="center",
                fontsize=fs, fontweight="bold", color=TEXT_DARK, zorder=6)
    if owner:
        ax.text(cx, cy - BH/2 + 0.10, f"[{owner}]",
                ha="center", va="center", fontsize=5.8,
                color="#4e342e", style="italic", zorder=6)

def draw_diamond(ax, cx, cy, label, owner=None, improved=False, fs=7.2):
    ec = IMPROVED_EC if improved else AMBER_DEC
    fc = IMPROVED_FC if improved else AMBER_BG
    hw, hh = DW/2, DH/2
    pts = [(cx, cy+hh), (cx+hw, cy), (cx, cy-hh), (cx-hw, cy)]
    ax.add_patch(Polygon(pts, closed=True, lw=1.8,
                 edgecolor=ec, facecolor=fc, zorder=5))
    lines = label.split("\n")
    n = len(lines)
    spacing = 0.135
    for i, ln in enumerate(lines):
        oy = (n - 1) * spacing / 2 - i * spacing
        oy += 0.06 if owner else 0
        ax.text(cx, cy + oy, ln, ha="center", va="center",
                fontsize=fs, fontweight="bold", color=TEXT_DARK, zorder=6)
    if owner:
        ax.text(cx, cy - hh + 0.10, f"[{owner}]",
                ha="center", va="center", fontsize=5.5,
                color="#4e342e", style="italic", zorder=6)

def arrow(ax, pts, col=DARK_ARR, lw=1.5):
    for i in range(len(pts) - 2):
        ax.plot([pts[i][0], pts[i+1][0]], [pts[i][1], pts[i+1][1]],
                color=col, lw=lw, solid_capstyle="butt", zorder=4)
    ax.annotate("", xy=pts[-1], xytext=pts[-2],
                arrowprops=dict(arrowstyle="-|>", color=col, lw=lw,
                                mutation_scale=13), zorder=4)

def alabel(ax, x, y, txt, col=DARK_ARR, fs=6.8):
    ax.text(x, y, txt, ha="center", va="center", fontsize=fs,
            fontweight="bold", color=col, zorder=9,
            bbox=dict(boxstyle="round,pad=0.2", facecolor=WHITE,
                      edgecolor=col, lw=0.8, alpha=1.0))

def terminal(ax, cx, cy, label, color):
    ax.add_patch(mpatches.Ellipse((cx, cy), 1.1, 0.36,
                 lw=1.8, edgecolor=color, facecolor=color, zorder=5))
    ax.text(cx, cy, label, ha="center", va="center",
            fontsize=7.5, fontweight="bold", color=WHITE, zorder=6)

def draw_diagram(ax, improved, rp_c, rp_i):
    W, H = 14.5, 14.0
    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.set_aspect("auto")
    ax.axis("off")
    ax.set_facecolor(WHITE)

    rp = rp_i if improved else rp_c

    TAB = 0.85
    lanes = [
        (0.0,  2.2,  "Stage 5", "Purchase\nOrder",           "Proc. Officer"),
        (2.2,  5.2,  "Stage 4", "Negotiations\n& Contracts",  "Contracts Mgr / Legal"),
        (5.2,  7.4,  "Stage 3", "Evaluation",                "Eval. Comm / Engineers"),
        (7.4, 11.2,  "Stage 2", "Bidding",                   "Proc. Team / Suppliers"),
        (11.2, 13.4, "Stage 1", "Purchasing\nRequest",        "Site Mgr / Proc. Officer"),
    ]
    lane_colors = ["#e8f5e9", "#f3e5f5", "#e3f2fd", "#fff8e1", "#fce4ec"]
    tab_colors  = ["#2e7d32", "#6a1b9a", "#1565c0", "#e65100", "#b71c1c"]

    for (yb, yt, stage_lbl, name_lbl, role_lbl), lc, tc in zip(lanes, lane_colors, tab_colors):
        lane_h = yt - yb

        # Background panel
        ax.add_patch(mpatches.Rectangle(
            (TAB, yb), W - TAB, lane_h,
            lw=0.6, edgecolor=GRAY_BORDER, facecolor=lc, zorder=0))

        # Coloured tab
        ax.add_patch(FancyBboxPatch(
            (0.03, yb + 0.05), TAB - 0.08, lane_h - 0.10,
            boxstyle="round,pad=0.04", lw=0, facecolor=tc, zorder=2))

        cx = TAB / 2  # horizontal centre of tab

        # ── FIX: position text using absolute y within the lane ──────────
        # All text is rotation=90, so y-axis = along the tab height.
        # Stage label near top, lane name in middle, role near bottom.

        # Stage label — top 20% of lane
        ax.text(cx, yb + lane_h * 0.82, stage_lbl,
                ha="center", va="center",
                fontsize=6.5, fontweight="bold",
                color=WHITE, rotation=90, zorder=3)

        # Lane name — centre of lane
        ax.text(cx, yb + lane_h * 0.50, name_lbl,
                ha="center", va="center",
                fontsize=5.8, fontweight="bold",
                color=WHITE, rotation=90, zorder=3)

        # Role — bottom 20% of lane
        ax.text(cx, yb + lane_h * 0.16, role_lbl,
                ha="center", va="center",
                fontsize=4.5, color=WHITE,
                rotation=90, zorder=3, alpha=0.92)

    # ── Title banner
    title_col = "#00695c" if improved else "#b71c1c"
    title_txt = ("TO-BE PROCESS  (Automated / Improved)"
                 if improved else "AS-IS PROCESS  (Current / Manual)")
    ax.add_patch(FancyBboxPatch((TAB + 0.1, 13.45), W - TAB - 0.2, 0.40,
                 boxstyle="round,pad=0.06", lw=1.8,
                 edgecolor=title_col, facecolor=WHITE, zorder=6))
    ax.text((TAB + W) / 2, 13.65, title_txt,
            ha="center", va="center", fontsize=10.5,
            fontweight="bold", color=title_col, zorder=7)

    X = [2.00, 3.90, 5.80, 7.70, 9.60, 11.50, 13.10]
    Y1  = 12.30
    Y2a = 10.00
    Y2b =  8.55
    Y3  =  6.30
    Y4  =  3.70
    Y5  =  1.10

    terminal(ax, X[0], Y1 + 0.55, "START", GREEN_DARK)
    arrow(ax, [(X[0], Y1 + 0.37), (X[0], Y1 + BH/2 + 0.02)])

    terminal(ax, X[2], Y5 - 0.52, "END", RED_ARR)
    arrow(ax, [(X[2], Y5 - BH/2 - 0.02), (X[2], Y5 - 0.34)], RED_ARR)

    # STAGE 1
    draw_box(ax, X[0], Y1, "Need for\nRaw Material", "Site Manager")
    draw_box(ax, X[2], Y1, "Send PR", "Proc. Officer")

    p_bud = int(rp["Budget\nApproval"] * 100)
    draw_diamond(ax, X[4], Y1,
                 f"Budget\nApproval?\n({p_bud}% reject)",
                 "Finance Dept", improved)

    draw_box(ax, X[6], Y1, "Specification\n& Criteria", "Proc. Officer")

    arrow(ax, [(X[0] + BW/2, Y1), (X[2] - BW/2, Y1)])
    arrow(ax, [(X[2] + BW/2, Y1), (X[4] - DW/2, Y1)])
    arrow(ax, [(X[4] + DW/2, Y1), (X[6] - BW/2, Y1)], GREEN_ARR)
    alabel(ax, (X[4] + DW/2 + X[6] - BW/2) / 2, Y1 + 0.22, "Yes", GREEN_ARR)

    loop_y1 = 11.22
    arrow(ax, [(X[4], Y1 - DH/2),
               (X[4], loop_y1),
               (X[2], loop_y1),
               (X[2], Y1 - BH/2)], RED_ARR)
    alabel(ax, (X[4] + X[2]) / 2, loop_y1 - 0.18,
           f"No ({p_bud}%) → Revise PR", RED_ARR)

    arrow(ax, [(X[0], Y1 - BH/2),
               (X[0], 11.22),
               (X[0], Y2a + BH/2)])

    arrow(ax, [(X[6], Y1 - BH/2),
               (X[6], 11.22),
               (X[6], Y2a + BH/2)])

    # STAGE 2
    draw_box(ax, X[0], Y2a, "PR Approved\nfor Bidding", "Proc. Officer")

    p_sho = int(rp["Shortlist\nSuppliers"] * 100)
    draw_diamond(ax, X[2], Y2a,
                 f"Shortlist\nSuppliers?\n({p_sho}% fail)",
                 "Proc. Team", improved)

    draw_box(ax, X[4], Y2a, "Supplier\nDatabase", "Proc. Team",
             fc=BLUE_NODE, ec=BLUE_EDGE)
    draw_box(ax, X[6], Y2a, "Preferred\nSupplier", "Proc. Manager",
             fc=BLUE_NODE, ec=BLUE_EDGE)

    rfq_label = "E-Source\n(Digital RFQ)" if improved else "RFQ\n(Request for Quote)"
    draw_box(ax, X[4], Y2b, rfq_label, "Proc. Officer", improved=improved)
    draw_box(ax, X[2], Y2b, "Quotations\nReceived", "Proc. Officer")

    arrow(ax, [(X[0] + BW/2, Y2a), (X[2] - DW/2, Y2a)])
    arrow(ax, [(X[2] + DW/2, Y2a), (X[4] - BW/2, Y2a)], GREEN_ARR)
    alabel(ax, (X[2] + DW/2 + X[4] - BW/2) / 2, Y2a + 0.22,
           f"Yes ({100 - p_sho}%)", GREEN_ARR)
    arrow(ax, [(X[4] + BW/2, Y2a), (X[6] - BW/2, Y2a)])

    arrow(ax, [(X[2], Y2a - DH/2), (X[2], Y2b + BH/2)], RED_ARR)
    alabel(ax, X[2] + 0.65, (Y2a - DH/2 + Y2b + BH/2) / 2,
           f"No ({p_sho}%)", RED_ARR)

    arrow(ax, [(X[4] - BW/2, Y2b), (X[2] + BW/2, Y2b)], DARK_ARR)
    alabel(ax, (X[4] - BW/2 + X[2] + BW/2) / 2, Y2b + 0.22,
           "Quotes received", DARK_ARR)

    arrow(ax, [(X[6], Y2a - BH/2),
               (X[6], Y2b),
               (X[4] + BW/2, Y2b)], RED_ARR)
    alabel(ax, X[6] - 0.40, (Y2a - BH/2 + Y2b) / 2, "No match\nin DB", RED_ARR)

    arrow(ax, [(X[2], Y2b - BH/2),
               (X[2], 7.42),
               (X[0], 7.42),
               (X[0], Y3 + BH/2)])

    # STAGE 3
    draw_box(ax, X[0], Y3, "Evaluation", "Eval Committee")
    draw_box(ax, X[2], Y3, "Technical\nReview", "Tech Engineer", improved=improved)
    draw_box(ax, X[4], Y3, "Financial\nReview", "Finance Analyst", improved=improved)

    p_qual = int(rp["Qualified\nSuppliers"] * 100)
    draw_diamond(ax, X[6], Y3,
                 f"Qualified\nSuppliers?\n({p_qual}% fail)",
                 "Proc. Manager", improved)

    arrow(ax, [(X[0] + BW/2, Y3), (X[2] - BW/2, Y3)])
    arrow(ax, [(X[2] + BW/2, Y3), (X[4] - BW/2, Y3)])
    arrow(ax, [(X[4] + BW/2, Y3), (X[6] - DW/2, Y3)])

    arrow(ax, [(X[6], Y3 - DH/2),
               (X[6], 5.22),
               (X[0], 5.22),
               (X[0], Y4 + BH/2)], GREEN_ARR)
    alabel(ax, (X[6] + X[0]) / 2, 5.22 + 0.20,
           f"Yes ({100 - p_qual}%) — proceed to Negotiations", GREEN_ARR)

    loop_y3 = 7.38
    arrow(ax, [(X[6], Y3 + DH/2),
               (X[6], loop_y3),
               (X[0], loop_y3),
               (X[0], Y3 + BH/2)], RED_ARR)
    alabel(ax, (X[6] + X[0]) / 2, loop_y3 + 0.18,
           f"No ({p_qual}%) → Re-evaluate suppliers", RED_ARR)

    # STAGE 4
    neg_label = "e-Negotiate\n(Digital)" if improved else "Negotiations"
    draw_box(ax, X[0], Y4, neg_label, "Contracts Mgr", improved=improved)

    p_neg = int(rp["Negotiations\nFinalized"] * 100)
    draw_diamond(ax, X[2], Y4,
                 f"Negotiations\nFinalized?\n({p_neg}% fail)",
                 "Contracts Mgr", improved)

    draw_box(ax, X[4], Y4, "Issue Contract\n& T&C", "Legal/Director")
    draw_box(ax, X[6], Y4, "Sign\nAgreement", "Director/PM")

    arrow(ax, [(X[0] + BW/2, Y4), (X[2] - DW/2, Y4)])
    arrow(ax, [(X[2] + DW/2, Y4), (X[4] - BW/2, Y4)], GREEN_ARR)
    alabel(ax, (X[2] + DW/2 + X[4] - BW/2) / 2, Y4 + 0.22, "Yes", GREEN_ARR)
    arrow(ax, [(X[4] + BW/2, Y4), (X[6] - BW/2, Y4)])

    loop_y4 = 2.22
    arrow(ax, [(X[2], Y4 - DH/2),
               (X[2], loop_y4),
               (X[0], loop_y4),
               (X[0], Y4 - BH/2)], RED_ARR)
    alabel(ax, (X[2] + X[0]) / 2, loop_y4 - 0.18,
           f"No ({p_neg}%) → Re-negotiate", RED_ARR)

    arrow(ax, [(X[6], Y4 - BH/2),
               (X[6], 2.22),
               (X[2], 2.22),
               (X[2], Y5 + BH/2)])

    # STAGE 5
    po_label = "Auto PO\n(2000 Units)" if improved else "Release PO\n(2000 Units)"
    draw_box(ax, X[2], Y5, po_label, "Proc. Officer",
             fc=BLUE_NODE, ec=BLUE_EDGE, improved=False)

    if improved:
        _bud_c = STAGES_C["Budget\nApproval"]["mu"]
        _bud_i = STAGES_I["Budget\nApproval"]["mu"]
        _bud_pct = int((_bud_c - _bud_i) / _bud_c * 100)
        callouts = [
            (X[4], Y1 + 0.62,  f"Auto Budget Check\n↓{_bud_pct}% cycle time"),
            (X[4], Y2b + 0.66, "Digital E-Sourcing\n↓50% cycle time"),
            (X[2], Y3 + 0.66,  "Parallel Reviews\n↓40% cycle time"),
            (X[0], Y4 + 0.66,  "e-Negotiate\n↓47% cycle time"),
        ]
        for cx, cy, ctxt in callouts:
            ax.text(cx, cy, ctxt, ha="center", va="center", fontsize=6.5,
                    fontweight="bold", color="#004d40",
                    bbox=dict(boxstyle="round,pad=0.25", facecolor="#e0f2f1",
                              edgecolor=IMPROVED_EC, lw=1.2), zorder=10)

        ax.add_patch(FancyBboxPatch((X[4] + 0.2, Y5 - 0.55), 2.2, 1.1,
                     boxstyle="round,pad=0.1", lw=2.5,
                     edgecolor=GREEN_DARK, facecolor=GREEN_LIGHT, zorder=11))
        ax.text(X[4] + 1.30, Y5 + 0.25,
                f"↓{savings_pct:.0f}%",
                ha="center", va="center", fontsize=16, fontweight="bold",
                color=GREEN_DARK, zorder=12)
        ax.text(X[4] + 1.30, Y5 - 0.18,
                "cycle time reduction",
                ha="center", va="center", fontsize=7.5, fontweight="bold",
                color=GREEN_DARK, zorder=12)

    df_use = si if improved else sc
    avg_tot = df_use.total.mean()
    std_tot = df_use.total.std()
    p95     = np.percentile(df_use.total, 95)
    rw_cols = [c for c in df_use.columns if c.startswith("rw_")]
    avg_rw  = df_use[rw_cols].sum(axis=1).mean()
    stats_txt = (f"Avg Cycle Time: {avg_tot:.0f} days   "
                 f"Std Dev: {std_tot:.0f} days   "
                 f"P95: {p95:.0f} days   "
                 f"Avg Rework Loops/Job: {avg_rw:.2f}")
    ax.text((TAB + W) / 2, 0.08, stats_txt, ha="center", va="bottom",
            fontsize=7.2, color="#424242",
            bbox=dict(boxstyle="round,pad=0.22", facecolor="#f5f5f5",
                      edgecolor=GRAY_BORDER, lw=0.8), zorder=8)

    leg = [
        mpatches.Patch(fc=WHITE,       ec=GREEN_DARK, lw=1.6, label="Process step"),
        mpatches.Patch(fc=IMPROVED_FC, ec=IMPROVED_EC, lw=1.6, label="Improved step (TO-BE)"),
        mpatches.Patch(fc=AMBER_BG,    ec=AMBER_DEC, lw=1.6, label="Decision gate"),
        mpatches.Patch(fc=BLUE_NODE,   ec=BLUE_EDGE, lw=1.6, label="Database / output"),
        mpatches.Patch(color=RED_ARR,  label="Reject / rework path"),
        mpatches.Patch(color=GREEN_ARR, label="Approved path"),
        mpatches.Patch(color=DARK_ARR,  label="Standard flow"),
    ]
    ax.legend(handles=leg, loc="lower right", fontsize=6.8,
              facecolor=WHITE, edgecolor=GRAY_BORDER,
              labelcolor=TEXT_DARK, ncol=4, framealpha=1.0,
              bbox_to_anchor=(1.01, 0.0))


fig, axes = plt.subplots(2, 1, figsize=(24, 30))
fig.patch.set_facecolor(WHITE)
fig.subplots_adjust(top=0.97, bottom=0.01, left=0.01, right=0.99, hspace=0.05)

fig.text(0.5, 0.988,
         "Five Stages of Procurement Process — AS-IS vs TO-BE Flow Diagrams",
         ha="center", fontsize=17, fontweight="bold", color="#1a1a1a")
fig.text(0.5, 0.982,
         "Construction Project Raw Material Procurement  |  Monte Carlo Simulation n=1,000",
         ha="center", fontsize=9, color="#555")

draw_diagram(axes[0], False, REWORK_C, REWORK_I)
draw_diagram(axes[1], True,  REWORK_C, REWORK_I)

for ax, lbl in zip(axes, ["(A) AS-IS — Current Manual Process", "(B) TO-BE — Improved / Automated Process"]):
    ax.text(0.5, -0.005, lbl, transform=ax.transAxes,
            ha="center", va="top", fontsize=9, color="#555", style="italic")

out = "/home/seanhegede/procurement_flow_FIXED.png"
fig.savefig(out, dpi=160, bbox_inches="tight", facecolor=WHITE)
print(f"Saved → {out}")
print(f"AS-IS mean: {sc.total.mean():.1f}d  |  TO-BE mean: {si.total.mean():.1f}d  |  Saving: {savings_pct:.1f}%")

/home/seanhegede/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Saved → /home/seanhegede/procurement_flow_FIXED.png
AS-IS mean: 139.7d  |  TO-BE mean: 93.8d  |  Saving: 32.9%


In [10]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# 1.  PROCESS STEP DEFINITIONS  (directly from procurement flow diagrams)
#
#  Columns: (name, mean_days, std_days, rho, rework_prob,
#             risk_event_prob, risk_penalty_mean_days)
#
#  rho         = 0.12 × mu   (base utilisation, λ=0.12 jobs/day)
#  rework_prob = from REWORK_C / REWORK_I decision gates (0 elsewhere)
#  risk_p      = min(0.20, (sigma/mu) × rho × 0.8)   higher variability → more risk
#  risk_days   = mu × 0.35                            penalty ≈ 35% of mean service
# ─────────────────────────────────────────────────────────────────────────────

AS_IS_STEPS = [
    # name                       mean  std    rho   rework  risk_p  risk_d
    ("Need for Raw Material",       3,   1,  0.36,  0.00,  0.096,   1.0),
    ("Send PR",                     5,   2,  0.60,  0.00,  0.192,   1.8),
    ("Budget Approval",            10,   4,  1.20,  0.30,  0.200,   3.5),
    ("Specification & Criteria",    4, 1.5,  0.48,  0.00,  0.144,   1.4),
    ("PR Approved for Bidding",     3,   1,  0.36,  0.00,  0.096,   1.0),
    ("Shortlist Suppliers",        12,   5,  1.44,  0.25,  0.200,   4.2),
    ("RFQ",                        14,   6,  1.68,  0.00,  0.200,   4.9),
    ("Quotations Received",         8,   3,  0.96,  0.00,  0.200,   2.8),
    ("Evaluation",                  7, 2.5,  0.84,  0.00,  0.200,   2.4),
    ("Technical Review",           10,   4,  1.20,  0.00,  0.200,   3.5),
    ("Financial Review",            9, 3.5,  1.08,  0.00,  0.200,   3.2),
    ("Qualified Suppliers",         5,   2,  0.60,  0.20,  0.192,   1.8),
    ("Negotiations",               15,   6,  1.80,  0.00,  0.200,   5.2),
    ("Negotiations Finalized",      8,   3,  0.96,  0.35,  0.200,   2.8),
    ("Issue Contract & T&C",        6,   2,  0.72,  0.00,  0.192,   2.1),
    ("Sign Agreement",              4, 1.5,  0.48,  0.00,  0.144,   1.4),
    ("Release PO (2000 Units)",     4, 1.5,  0.48,  0.00,  0.144,   1.4),
]

TO_BE_STEPS = [
    # name                       mean  std    rho   rework  risk_p  risk_d
    ("Need for Raw Material",       3,   1,  0.36,  0.00,  0.060,   0.8),
    ("Send PR",                     5,   2,  0.60,  0.00,  0.100,   1.2),
    ("Budget Approval (Auto)",      5,   2,  0.60,  0.10,  0.100,   1.2),
    ("Specification & Criteria",    4, 1.5,  0.48,  0.00,  0.090,   1.0),
    ("PR Approved for Bidding",     3,   1,  0.36,  0.00,  0.060,   0.8),
    ("Shortlist Suppliers (e-Src)", 5,   2,  0.60,  0.08,  0.100,   1.2),
    ("RFQ (e-Sourcing)",            7,   2,  0.84,  0.00,  0.100,   1.8),
    ("Quotations Received",         8,   3,  0.96,  0.00,  0.100,   2.0),
    ("Evaluation",                  7, 2.5,  0.84,  0.00,  0.100,   1.8),
    ("Technical Review (Parallel)", 6,   2,  0.72,  0.00,  0.100,   1.5),
    ("Financial Review (Parallel)", 6,   2,  0.72,  0.00,  0.100,   1.5),
    ("Qualified Suppliers",         5,   2,  0.60,  0.07,  0.100,   1.2),
    ("e-Negotiate",                 8,   3,  0.96,  0.00,  0.100,   2.0),
    ("Negotiations Finalized",      8,   3,  0.96,  0.12,  0.100,   2.0),
    ("Issue Contract & T&C",        6,   2,  0.72,  0.00,  0.100,   1.5),
    ("Sign Agreement (e-Sign)",     2,   1,  0.24,  0.00,  0.060,   0.5),
    ("Auto PO (2000 Units)",        4, 1.5,  0.48,  0.00,  0.090,   1.0),
]

# ─────────────────────────────────────────────────────────────────────────────
# 2.  PERT / BETA SAMPLING  (unchanged from template)
# ─────────────────────────────────────────────────────────────────────────────

def pert_params(mean, std):
    lo  = max(0.5, mean - 3 * std)
    hi  = mean + 3 * std
    rng = hi - lo
    if rng < 1e-6:
        return None, None, mean, 0
    mu_norm  = np.clip((mean - lo) / rng, 0.01, 0.99)
    var_norm = np.clip((std / rng) ** 2, 1e-6, mu_norm * (1 - mu_norm) - 1e-6)
    common   = mu_norm * (1 - mu_norm) / var_norm - 1
    alpha    = max(mu_norm * common, 0.5)
    beta_    = max((1 - mu_norm) * common, 0.5)
    return alpha, beta_, lo, rng

def sample_duration(mean, std, n):
    alpha, beta_, loc, scale = pert_params(mean, std)
    if scale == 0:
        return np.full(n, mean)
    return stats.beta.rvs(alpha, beta_, loc=loc, scale=scale, size=n)

def queue_wait(rho, mean_service):
    """M/G/1 Pollaczek-Khinchine approximation, capped at 3× mean service."""
    rho  = np.clip(rho, 0.0, 0.98)
    wait = (rho ** 2 * mean_service) / (2 * (1 - rho))
    return np.clip(wait, 0, 3 * mean_service)

# ─────────────────────────────────────────────────────────────────────────────
# 3.  MONTE CARLO ENGINE  (unchanged from template)
# ─────────────────────────────────────────────────────────────────────────────

def run_monte_carlo(steps, n_contracts=500, n_simulations=2000, seed=42):
    rng = np.random.default_rng(seed)
    N, S, K = n_contracts, n_simulations, len(steps)

    all_totals     = np.zeros((S, N))
    step_contrib   = np.zeros((S, K))
    bottleneck_mat = np.zeros((S, K))

    for sim in range(S):
        contract_days = np.zeros(N)
        for k, (name, mean, std, rho, rework_p, risk_p, risk_mu) in enumerate(steps):
            service = sample_duration(mean, std, N)
            if rework_p > 0:
                mask    = rng.random(N) < rework_p
                n_rw    = mask.sum()
                if n_rw > 0:
                    service[mask] += (sample_duration(mean, std, n_rw)
                                      + rng.uniform(1, 3, n_rw))
            qw       = queue_wait(rho, mean)
            qw_s     = qw * rng.uniform(0.7, 1.3, N)
            service += qw_s
            risk_mask = rng.random(N) < risk_p
            n_risk    = risk_mask.sum()
            if n_risk > 0:
                service[risk_mask] += rng.lognormal(
                    np.log(max(risk_mu, 1)), 0.4, n_risk)
            contract_days        += service
            step_contrib[sim, k]  = service.mean()
            bottleneck_mat[sim, k]= qw_s.mean()
        all_totals[sim] = contract_days

    flat = all_totals.flatten()
    return {
        'all_totals':      all_totals,
        'flat':            flat,
        'step_contrib':    step_contrib.mean(axis=0),
        'bottleneck_days': bottleneck_mat.mean(axis=0),
        'step_names':      [s[0] for s in steps],
        'p10':  np.percentile(flat, 10),
        'p50':  np.percentile(flat, 50),
        'p80':  np.percentile(flat, 80),
        'p90':  np.percentile(flat, 90),
        'p95':  np.percentile(flat, 95),
        'mean': flat.mean(),
    }

# ─────────────────────────────────────────────────────────────────────────────
# 4.  RUN BOTH SCENARIOS
# ─────────────────────────────────────────────────────────────────────────────

print("Running Monte Carlo — AS-IS  (2,000 sims × 500 contracts)...")
mc_b = run_monte_carlo(AS_IS_STEPS, n_contracts=500, n_simulations=2000, seed=42)

print("Running Monte Carlo — TO-BE  (2,000 sims × 500 contracts)...")
mc_a = run_monte_carlo(TO_BE_STEPS, n_contracts=500, n_simulations=2000, seed=99)

print("Done. Building dashboard...\n")

# ─────────────────────────────────────────────────────────────────────────────
# 5.  COLOURS & STYLE
# ─────────────────────────────────────────────────────────────────────────────

RED   = '#c0392b'
GRN   = '#27ae60'
AMBER = '#e67e22'
BLUE  = '#2980b9'
DARK  = '#2c3e50'
LGREY = '#ecf0f1'
WHITE = '#ffffff'
BORD  = '#bdc3c7'

def style_ax(ax, title, xlabel='', ylabel=''):
    ax.set_facecolor(WHITE)
    ax.set_title(title, fontweight='bold', fontsize=9.5, pad=6,
                 color=DARK, loc='left')
    if xlabel: ax.set_xlabel(xlabel, fontsize=8, color=DARK)
    if ylabel: ax.set_ylabel(ylabel, fontsize=8, color=DARK)
    ax.tick_params(labelsize=7.5)
    ax.grid(True, alpha=0.22, linestyle='--')
    for sp in ax.spines.values():
        sp.set_color(BORD)

# ─────────────────────────────────────────────────────────────────────────────
# 6.  FIGURE LAYOUT  (same 3×3 grid as template)
# ─────────────────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(20, 15))
fig.patch.set_facecolor(LGREY)

gs = GridSpec(3, 3, figure=fig,
              hspace=0.52, wspace=0.38,
              left=0.06, right=0.97, top=0.91, bottom=0.06)

ax_hist  = fig.add_subplot(gs[0, :2])   # schedule distribution histogram
ax_kpi   = fig.add_subplot(gs[0, 2])    # KPI summary panel
ax_stp_b = fig.add_subplot(gs[1, 0])    # step breakdown AS-IS
ax_stp_a = fig.add_subplot(gs[1, 1])    # step breakdown TO-BE
ax_bott  = fig.add_subplot(gs[1, 2])    # bottleneck queue wait comparison
ax_scurv = fig.add_subplot(gs[2, :2])   # S-curve CDF
ax_risk  = fig.add_subplot(gs[2, 2])    # schedule risk tornado (AS-IS)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 1 — Schedule Distribution Histogram
# ─────────────────────────────────────────────────────────────────────────────

max_x = max(mc_b['p95'], mc_a['p95']) * 1.10
bins  = np.linspace(0, max_x, 90)

ax_hist.hist(mc_b['flat'], bins=bins, density=True,
             alpha=0.50, color=RED, label='AS-IS')
ax_hist.hist(mc_a['flat'], bins=bins, density=True,
             alpha=0.50, color=GRN, label='TO-BE')

for mc, col in [(mc_b, RED), (mc_a, GRN)]:
    ax_hist.axvline(mc['p50'], color=col, lw=2.0, ls='-',  alpha=0.9)
    ax_hist.axvline(mc['p90'], color=col, lw=1.5, ls='--', alpha=0.75)
    ax_hist.axvline(mc['p10'], color=col, lw=1.0, ls=':',  alpha=0.60)

ax_hist.axvspan(mc_b['p10'], mc_b['p90'], alpha=0.07, color=RED,
                label='AS-IS P10–P90 band')
ax_hist.axvspan(mc_a['p10'], mc_a['p90'], alpha=0.09, color=GRN,
                label='TO-BE P10–P90 band')

style_ax(ax_hist,
         'Schedule Duration Distribution  —  Monte Carlo (2,000 runs × 500 contracts)',
         'Total Procurement Duration (days)', 'Probability Density')

ymax = ax_hist.get_ylim()[1]
for mc, col, yf in [(mc_b, RED, 0.72), (mc_a, GRN, 0.50)]:
    ax_hist.annotate(
        f"P50 = {mc['p50']:.0f}d\nP90 = {mc['p90']:.0f}d",
        xy=(mc['p50'], 0), xytext=(mc['p50'], ymax * yf),
        fontsize=7.5, color=col, ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc=WHITE, ec=col, lw=1.2),
        arrowprops=dict(arrowstyle='->', color=col, lw=1.2)
    )

ax_hist.legend(fontsize=7.5, loc='upper right',
               facecolor=WHITE, edgecolor=BORD, framealpha=0.95)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 2 — KPI Panel
# ─────────────────────────────────────────────────────────────────────────────

ax_kpi.set_facecolor(DARK)
ax_kpi.set_xlim(0, 1); ax_kpi.set_ylim(0, 1)
for sp in ax_kpi.spines.values(): sp.set_visible(False)
ax_kpi.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

p50_sav = mc_b['p50'] - mc_a['p50']
p90_sav = mc_b['p90'] - mc_a['p90']
pct_sav = p50_sav / mc_b['p50'] * 100
rf_b    = mc_b['p90'] / mc_b['p50']
rf_a    = mc_a['p90'] / mc_a['p50']

kpis = [
    ("P50 DURATION\nAS-IS",         f"{mc_b['p50']:.0f} days",  '#e74c3c'),
    ("P50 DURATION\nTO-BE",         f"{mc_a['p50']:.0f} days",  '#2ecc71'),
    ("P50 SAVING",                  f"{p50_sav:.0f} days",       '#3498db'),
    ("P90 SAVING",                  f"{p90_sav:.0f} days",       '#9b59b6'),
    ("SCHEDULE\nCOMPRESSION",       f"{pct_sav:.1f}%",           '#f39c12'),
    ("RISK FACTOR\nP90/P50 AS-IS",  f"{rf_b:.2f}×",             '#e67e22'),
]

ax_kpi.text(0.5, 0.97, 'MONTE CARLO  KPIs',
            ha='center', va='top', fontsize=9, fontweight='bold',
            color=WHITE, transform=ax_kpi.transAxes)

for idx, (lbl, val, clr) in enumerate(kpis):
    col_k = idx % 2
    row_k = idx // 2
    xc = 0.25 + col_k * 0.50
    yc = 0.82 - row_k * 0.27
    ax_kpi.text(xc, yc,        val, ha='center', va='center',
                fontsize=15, fontweight='bold', color=clr,
                transform=ax_kpi.transAxes)
    ax_kpi.text(xc, yc - 0.09, lbl, ha='center', va='center',
                fontsize=6.8, color='#95a5a6',
                transform=ax_kpi.transAxes, multialignment='center')

# ─────────────────────────────────────────────────────────────────────────────
# CHARTS 3 & 4 — Step Duration Breakdown (service + queue wait stacked)
# ─────────────────────────────────────────────────────────────────────────────

def step_bars(ax, mc, color, label):
    names    = mc['step_names']
    contrib  = mc['step_contrib']
    qwait    = mc['bottleneck_days']
    service  = contrib - qwait
    x = np.arange(len(names))
    w = 0.60
    ax.bar(x, service, w, color=color,  alpha=0.85, label='Service time',
           edgecolor=WHITE)
    ax.bar(x, qwait,   w, bottom=service, color=AMBER, alpha=0.82,
           label='Queue wait', edgecolor=WHITE)
    for i, (s, q) in enumerate(zip(service, qwait)):
        ax.text(i, s + q + 0.3, f'{s+q:.1f}d',
                ha='center', va='bottom', fontsize=6, fontweight='bold',
                color=DARK)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=40, ha='right', fontsize=6.2)
    ax.legend(fontsize=7, loc='upper left',
              facecolor=WHITE, edgecolor=BORD, framealpha=0.95)
    style_ax(ax, f'Step Duration Breakdown — {label}',
             ylabel='Avg Days per Contract')

step_bars(ax_stp_b, mc_b, RED, 'AS-IS')
step_bars(ax_stp_a, mc_a, GRN, 'TO-BE')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 5 — Bottleneck Queue Wait Comparison (AS-IS vs TO-BE side by side)
# ─────────────────────────────────────────────────────────────────────────────

# Both scenarios share the same 17 stages in the same order
xb = np.arange(len(AS_IS_STEPS))
wb = 0.38
b_wait = mc_b['bottleneck_days']
a_wait = mc_a['bottleneck_days']

bars_b = ax_bott.bar(xb - wb/2, b_wait, wb,
                     color=RED, alpha=0.85, label='AS-IS', edgecolor=WHITE)
bars_a = ax_bott.bar(xb + wb/2, a_wait, wb,
                     color=GRN, alpha=0.85, label='TO-BE',  edgecolor=WHITE)

for bar in list(bars_b) + list(bars_a):
    h = bar.get_height()
    if h > 0.15:
        ax_bott.text(bar.get_x() + bar.get_width()/2, h + 0.05,
                     f'{h:.1f}', ha='center', va='bottom', fontsize=6,
                     color=DARK)

short_names = [s[0].split('(')[0].strip()[:14] for s in AS_IS_STEPS]
ax_bott.set_xticks(xb)
ax_bott.set_xticklabels(short_names, rotation=40, ha='right', fontsize=6.2)
style_ax(ax_bott,
         'Avg Queue Wait Days by Stage\n(Bottleneck Impact)',
         ylabel='Days waiting in queue')
ax_bott.legend(fontsize=7.5, loc='upper right',
               facecolor=WHITE, edgecolor=BORD, framealpha=0.95)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 6 — S-Curve (CDF comparison)
# ─────────────────────────────────────────────────────────────────────────────

for mc, col, lbl in [(mc_b, RED, 'AS-IS'), (mc_a, GRN, 'TO-BE')]:
    s = np.sort(mc['flat'])
    c = np.linspace(0, 1, len(s))
    ax_scurv.plot(s, c * 100, color=col, lw=2.5, label=lbl)

for pct, ls in [(50, '-'), (80, '--'), (90, ':')]:
    ax_scurv.axhline(pct, color='#7f8c8d', lw=0.8, ls=ls, alpha=0.55)

ax_scurv.axvspan(mc_b['p80'], mc_b['p90'],
                 alpha=0.08, color=RED, label='AS-IS risk zone P80–P90')

style_ax(ax_scurv,
         'Cumulative Schedule Probability — S-Curve',
         'Total Procurement Duration (days)', 'Contracts Completed (%)')
ax_scurv.set_xlim(left=0)
ax_scurv.set_ylim(0, 102)

# P-percentile labels — placed after xlim is set so get_xlim() is reliable
xmax = ax_scurv.get_xlim()[1]
for pct, ls in [(50, '-'), (80, '--'), (90, ':')]:
    ax_scurv.text(xmax * 0.995, pct + 0.8, f'P{pct}',
                  fontsize=7, color='#7f8c8d', va='bottom', ha='right')

ax_scurv.legend(fontsize=8, loc='lower right',
                facecolor=WHITE, edgecolor=BORD, framealpha=0.95)

# ─────────────────────────────────────────────────────────────────────────────
# CHART 7 — Schedule Risk Tornado  (AS-IS Pearson correlation)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(99)
N_sens = 5000
step_samples = []
total_sens   = np.zeros(N_sens)

for name, mean, std, rho, rework_p, risk_p, risk_mu in AS_IS_STEPS:
    s = sample_duration(mean, std, N_sens)
    if rework_p > 0:
        mask     = np.random.random(N_sens) < rework_p
        n_rw     = mask.sum()
        if n_rw > 0:
            s[mask] += (sample_duration(mean, std, n_rw)
                        + np.random.uniform(1, 3, n_rw))
    qw = queue_wait(rho, mean) * np.random.uniform(0.7, 1.3, N_sens)
    s += qw
    step_samples.append(s)
    total_sens += s

corrs      = [np.corrcoef(ss, total_sens)[0, 1] for ss in step_samples]
step_names = [s[0] for s in AS_IS_STEPS]

order        = np.argsort(np.abs(corrs))
sorted_names = [step_names[i] for i in order]
sorted_corrs = [corrs[i]      for i in order]
colors_t     = [RED if c > 0 else GRN for c in sorted_corrs]

y_pos = np.arange(len(sorted_names))
ax_risk.barh(y_pos, sorted_corrs, color=colors_t,
             edgecolor=WHITE, alpha=0.85, height=0.7)
ax_risk.set_yticks(y_pos)
ax_risk.set_yticklabels(sorted_names, fontsize=7.5)
ax_risk.axvline(0, color=DARK, lw=1.0)

for i, c in enumerate(sorted_corrs):
    ax_risk.text(c + (0.01 if c >= 0 else -0.01), i,
                 f'{c:.2f}', va='center',
                 ha='left' if c >= 0 else 'right', fontsize=7)

style_ax(ax_risk,
         'Schedule Risk Tornado — AS-IS\n(Pearson r with total duration)',
         xlabel='Correlation coefficient')
ax_risk.set_xlim(-0.1, 1.05)
ax_risk.legend(handles=[
    mpatches.Patch(color=RED, alpha=0.85, label='Increases total duration'),
], fontsize=7, loc='lower right',
   facecolor=WHITE, edgecolor=BORD, framealpha=0.95)

# ─────────────────────────────────────────────────────────────────────────────
# SUPER-TITLE
# ─────────────────────────────────────────────────────────────────────────────

fig.suptitle(
    "Construction Project Raw Material Procurement  —  "
    "Monte Carlo Schedule Risk Dashboard\n"
    f"AS-IS  P50={mc_b['p50']:.0f}d  P90={mc_b['p90']:.0f}d     "
    f"TO-BE  P50={mc_a['p50']:.0f}d  P90={mc_a['p90']:.0f}d     "
    f"Schedule compression: {pct_sav:.1f}%     "
    f"n = 2,000 simulations × 500 contracts",
    fontsize=12, fontweight='bold', color=DARK, y=0.975
)

# ─────────────────────────────────────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────────────────────────────────────

out = '/home/seanhegede/procurement_monte_carlo.png'
fig.savefig(out, dpi=160, bbox_inches='tight',
            facecolor=fig.get_facecolor())
print(f"\nSaved → {out}")

print("\n" + "=" * 62)
print("MONTE CARLO SCHEDULE RISK  —  SUMMARY")
print("=" * 62)
print(f"\n  {'':35s}  {'AS-IS':>8}  {'TO-BE':>8}  {'Saving':>8}")
print(f"  {'-'*63}")
for lbl, b, a in [
    ("P10  best case (days)",         mc_b['p10'],  mc_a['p10']),
    ("P50  median (days)",            mc_b['p50'],  mc_a['p50']),
    ("P80  (days)",                   mc_b['p80'],  mc_a['p80']),
    ("P90  near-worst case (days)",   mc_b['p90'],  mc_a['p90']),
    ("Mean (days)",                   mc_b['mean'], mc_a['mean']),
]:
    print(f"  {lbl:35s}  {b:>8.1f}  {a:>8.1f}  {b-a:>+8.1f}")

print(f"\n  Schedule compression (P50):  {pct_sav:.1f}%")
print(f"  Risk factor P90/P50  AS-IS:  {rf_b:.2f}×  →  TO-BE: {rf_a:.2f}×")
print(f"\n  Top 3 schedule risk drivers (AS-IS):")
for i in range(1, 4):
    idx = order[-i]
    print(f"    {i}. {step_names[idx]:30s}  r = {corrs[idx]:.3f}")
print("=" * 62)

Running Monte Carlo — AS-IS  (2,000 sims × 500 contracts)...
Running Monte Carlo — TO-BE  (2,000 sims × 500 contracts)...
Done. Building dashboard...


Saved → /home/seanhegede/procurement_monte_carlo.png

MONTE CARLO SCHEDULE RISK  —  SUMMARY

                                          AS-IS     TO-BE    Saving
  ---------------------------------------------------------------
  P10  best case (days)                   403.5     211.7    +191.7
  P50  median (days)                      434.5     228.2    +206.3
  P80  (days)                             455.5     239.3    +216.1
  P90  near-worst case (days)             466.5     245.3    +221.3
  Mean (days)                             434.8     228.4    +206.4

  Schedule compression (P50):  47.5%
  Risk factor P90/P50  AS-IS:  1.07×  →  TO-BE: 1.07×

  Top 3 schedule risk drivers (AS-IS):
    1. Shortlist Suppliers             r = 0.428
    2. Negotiations                    r = 0.413
    3. RFQ                             r = 0.394


In [11]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.gridspec import GridSpec
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# PARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
STAGES_C = {
    "Need for\nRaw Material":    dict(mu=3,  sigma=1,   owner="Site Mgr",      stage=1),
    "Send PR":                   dict(mu=5,  sigma=2,   owner="Proc. Officer", stage=1),
    "Budget\nApproval":          dict(mu=10, sigma=4,   owner="Finance Dept",  stage=1),
    "Specification\n& Criteria": dict(mu=4,  sigma=1.5, owner="Proc. Officer", stage=1),
    "PR Approved\nfor Bidding":  dict(mu=3,  sigma=1,   owner="Proc. Officer", stage=2),
    "Shortlist\nSuppliers":      dict(mu=12, sigma=5,   owner="Proc. Team",    stage=2),
    "RFQ":                       dict(mu=14, sigma=6,   owner="Proc. Officer", stage=2),
    "Quotations\nReceived":      dict(mu=8,  sigma=3,   owner="Proc. Officer", stage=2),
    "Evaluation":                dict(mu=7,  sigma=2.5, owner="Eval Comm.",    stage=3),
    "Technical\nReview":         dict(mu=10, sigma=4,   owner="Tech Eng.",     stage=3),
    "Financial\nReview":         dict(mu=9,  sigma=3.5, owner="Fin. Analyst",  stage=3),
    "Qualified\nSuppliers":      dict(mu=5,  sigma=2,   owner="Proc. Mgr",     stage=3),
    "Negotiations":              dict(mu=15, sigma=6,   owner="Contracts Mgr", stage=4),
    "Negotiations\nFinalized":   dict(mu=8,  sigma=3,   owner="Contracts Mgr", stage=4),
    "Issue Contract\n& T&C":     dict(mu=6,  sigma=2,   owner="Legal/Dir.",    stage=4),
    "Sign\nAgreement":           dict(mu=4,  sigma=1.5, owner="Director/PM",   stage=4),
    "Release PO\n(2000 Units)":  dict(mu=4,  sigma=1.5, owner="Proc. Officer", stage=5),
}
STAGES_I = {k: dict(v) for k, v in STAGES_C.items()}
STAGES_I["Budget\nApproval"].update(mu=5,  sigma=2)
STAGES_I["Shortlist\nSuppliers"].update(mu=5,  sigma=2)
STAGES_I["RFQ"].update(mu=7,  sigma=2)
STAGES_I["Technical\nReview"].update(mu=6,  sigma=2)
STAGES_I["Financial\nReview"].update(mu=6,  sigma=2)
STAGES_I["Negotiations"].update(mu=8,  sigma=3)
STAGES_I["Sign\nAgreement"].update(mu=2,  sigma=1)

REWORK_C = {"Budget\nApproval":0.30, "Shortlist\nSuppliers":0.25,
            "Qualified\nSuppliers":0.20, "Negotiations\nFinalized":0.35}
REWORK_I = {"Budget\nApproval":0.10, "Shortlist\nSuppliers":0.08,
            "Qualified\nSuppliers":0.07, "Negotiations\nFinalized":0.12}

LAM = 0.12   # arrival rate: procurement jobs per day (≈1 job every 8 working days)

# ─────────────────────────────────────────────────────────────────────────────
# SIMULATION
# ─────────────────────────────────────────────────────────────────────────────
def simulate(stages, rework, n=2000, seed=42):
    rng = np.random.default_rng(seed)
    recs = []
    for _ in range(n):
        tot, st, rw = 0., {}, {}
        for s, p in stages.items():
            svc = max(0.5, rng.normal(p["mu"], p["sigma"]))
            extra, loops = 0., 0
            if s in rework:
                while rng.random() < rework[s]:
                    loops += 1
                    extra += max(0.5, rng.normal(p["mu"], p["sigma"])) * 0.8
            st[s] = svc + extra
            rw[s] = loops
            tot  += svc + extra
        rec = {"total": tot}
        rec.update({f"t_{s}": v for s, v in st.items()})
        rec.update({f"rw_{s}": rw.get(s, 0) for s in rework})
        recs.append(rec)
    return pd.DataFrame(recs)

sc = simulate(STAGES_C, REWORK_C, seed=42)
si = simulate(STAGES_I, REWORK_I, seed=99)
snames  = list(STAGES_C.keys())
slabels = [s.replace("\n", " ") for s in snames]
savings_pct = (1 - si["total"].mean() / sc["total"].mean()) * 100

# ─────────────────────────────────────────────────────────────────────────────
# M/G/1 QUEUEING — Pollaczek-Khinchine mean waiting time
# Wq = ρ/(1-ρ) · E[S]/2 · (1 + Cs²)
# where Cs² = Var[S]/E[S]²  (squared coefficient of variation)
# Valid only when ρ < 1; saturated stages (ρ≥1) have Wq → ∞
# ─────────────────────────────────────────────────────────────────────────────
def mg1_metrics(df, stages):
    rows = []
    for s in stages:
        ES   = df[f"t_{s}"].mean()
        Var  = df[f"t_{s}"].var()
        Cs2  = Var / ES**2
        rho  = LAM * ES
        if rho < 1.0:
            Wq = (rho / (1 - rho)) * (ES / 2) * (1 + Cs2)
            Lq = LAM * Wq
        else:
            Wq = np.inf
            Lq = np.inf
        rows.append(dict(stage=s, label=s.replace("\n"," "),
                         ES=ES, Var=Var, Cs2=Cs2, CoV=np.sqrt(Cs2),
                         rho=rho, Wq=Wq, Lq=Lq))
    return pd.DataFrame(rows)

mq_c = mg1_metrics(sc, snames)
mq_i = mg1_metrics(si, snames)

# ─────────────────────────────────────────────────────────────────────────────
# SENSITIVITY: one-at-a-time — improve one stage, keep all others AS-IS
# ─────────────────────────────────────────────────────────────────────────────
def sensitivity_ota():
    baseline = sc["total"].mean()
    results  = []
    for s in snames:
        # build a mixed stages dict: this stage = TO-BE, all others = AS-IS
        mixed = {k: dict(v) for k, v in STAGES_C.items()}
        mixed[s] = dict(STAGES_I[s])
        # rework: use TO-BE rework prob for this stage if it has one
        mixed_rw = dict(REWORK_C)
        if s in REWORK_I:
            mixed_rw[s] = REWORK_I[s]
        df_mix = simulate(mixed, mixed_rw, n=2000, seed=42)
        delta  = baseline - df_mix["total"].mean()
        results.append(dict(stage=s, label=s.replace("\n"," "), delta=delta))
    return pd.DataFrame(results).sort_values("delta", ascending=True)

df_sens = sensitivity_ota()

# ─────────────────────────────────────────────────────────────────────────────
# COLOURS & HELPERS
# ─────────────────────────────────────────────────────────────────────────────
C_RED    = "#e53935"
C_AMB    = "#fb8c00"
C_GRN    = "#2e7d32"
C_GRN2   = "#66bb6a"
C_BLU    = "#1565c0"
C_GRY    = "#757575"
C_LGRY   = "#f5f5f5"
C_WHITE  = "#ffffff"
C_BORD   = "#e0e0e0"
C_SAT    = "#b71c1c"   # saturated stage fill

def rho_color(rho):
    if rho >= 1.0: return C_RED
    if rho >= 0.8: return C_AMB
    return C_GRN2

def sty(ax, title, xl="", yl="", grid_axis="y"):
    ax.set_facecolor("#fafafa")
    for sp in ax.spines.values():
        sp.set_edgecolor(C_BORD)
    ax.tick_params(colors="#333", labelsize=8)
    ax.xaxis.label.set_color("#333")
    ax.yaxis.label.set_color("#333")
    ax.set_title(title, fontsize=9.5, fontweight="bold",
                 color="#1a1a1a", pad=7, loc="left")
    ax.grid(axis=grid_axis, color=C_BORD, lw=0.7, ls="--", zorder=0)
    if xl: ax.set_xlabel(xl, fontsize=8.5)
    if yl: ax.set_ylabel(yl, fontsize=8.5)

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE
# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(26, 24))
fig.patch.set_facecolor(C_WHITE)
gs = GridSpec(3, 3, figure=fig,
              hspace=0.62, wspace=0.40,
              left=0.06, right=0.97, top=0.93, bottom=0.04)

ax1 = fig.add_subplot(gs[0, 0])   # cycle time distribution
ax2 = fig.add_subplot(gs[0, 1])   # bottleneck map (utilisation)
ax3 = fig.add_subplot(gs[0, 2])   # M/G/1 queue waiting time
ax4 = fig.add_subplot(gs[1, 0])   # rework cost per gate
ax5 = fig.add_subplot(gs[1, 1])   # improvement waterfall
ax6 = fig.add_subplot(gs[1, 2])   # process variability (CoV)
ax7 = fig.add_subplot(gs[2, :2])  # sensitivity tornado (left 2/3)
ax8 = fig.add_subplot(gs[2, 2])   # KPI impact panel (right 1/3)

fig.text(0.5, 0.965,
         "Construction Project Procurement — Systems Engineering Analytics Dashboard",
         ha="center", fontsize=15, fontweight="bold", color="#1a1a1a")
fig.text(0.5, 0.952,
         (f"DES flow simulation  n=2,000 jobs  ·  M/G/1 queueing (Pollaczek-Khinchine)  ·  "
          f"λ = {LAM} jobs/day  ·  AS-IS → TO-BE  ·  ↓{savings_pct:.0f}% mean cycle time"),
         ha="center", fontsize=9, color="#555")

# ═══════════════════════════════════════════════════════════════════════════
# CHART 1 — Cycle Time Distribution
# ═══════════════════════════════════════════════════════════════════════════
bins = np.linspace(40, 400, 60)
ax1.hist(sc["total"], bins=bins, color=C_RED,  alpha=0.45, density=True, zorder=2)
ax1.hist(si["total"], bins=bins, color=C_GRN,  alpha=0.45, density=True, zorder=2)

# KDE smooth curves
for data, col in [(sc["total"], C_RED), (si["total"], C_GRN)]:
    kde = gaussian_kde(data, bw_method=0.15)
    xs  = np.linspace(data.min(), data.max(), 400)
    ax1.plot(xs, kde(xs), color=col, lw=2.2, zorder=4)

# Mean and P95 lines
for data, col in [(sc["total"], C_RED), (si["total"], C_GRN)]:
    ax1.axvline(data.mean(), color=col, lw=2.0, ls="--", zorder=5)
    ax1.axvline(np.percentile(data, 95), color=col, lw=1.3, ls=":", zorder=5)

sty(ax1, "1 · Procurement Cycle Time Distribution",
    "Total Duration (days)", "Probability Density")

ymax = ax1.get_ylim()[1]
ax1.text(sc["total"].mean()+3, ymax*0.55, f"{sc['total'].mean():.0f}d mean",
         color=C_RED, fontsize=7.5, fontweight="bold")
ax1.text(si["total"].mean()+3, ymax*0.35, f"{si['total'].mean():.0f}d mean",
         color=C_GRN, fontsize=7.5, fontweight="bold")

# Legend outside plot area — top right corner, no overlap
ax1.legend(handles=[
    mpatches.Patch(fc=C_RED, alpha=0.6,
        label=f"AS-IS   μ={sc['total'].mean():.0f}d  σ={sc['total'].std():.0f}d  P95={np.percentile(sc['total'],95):.0f}d"),
    mpatches.Patch(fc=C_GRN, alpha=0.6,
        label=f"TO-BE  μ={si['total'].mean():.0f}d  σ={si['total'].std():.0f}d  P95={np.percentile(si['total'],95):.0f}d"),
    plt.Line2D([0],[0], color="#555", lw=1.5, ls="--", label="— mean"),
    plt.Line2D([0],[0], color="#555", lw=1.3, ls=":",  label="··· P95"),
], fontsize=7.2, loc="upper center", framealpha=0.95,
   facecolor=C_WHITE, edgecolor=C_BORD)

ax1.text(0.04, 0.96,
         f"Saving: ↓{savings_pct:.0f}%\n−{sc['total'].mean()-si['total'].mean():.0f} days",
         transform=ax1.transAxes, fontsize=9, fontweight="bold", color=C_GRN,
         va="top", bbox=dict(boxstyle="round,pad=0.3", fc="#e8f5e9", ec=C_GRN, lw=1.2))

# ═══════════════════════════════════════════════════════════════════════════
# CHART 2 — Bottleneck Map: Stage Utilisation ρ = λ·E[S]
# Sorted descending by AS-IS ρ. Colour-coded: red=saturated, amber=high, green=ok
# ═══════════════════════════════════════════════════════════════════════════
rho_c_vals = mq_c["rho"].values
rho_i_vals = mq_i["rho"].values

# Sort by AS-IS rho descending
sort_idx = np.argsort(rho_c_vals)[::-1]
sorted_labels = [slabels[i] for i in sort_idx]
sorted_rho_c  = rho_c_vals[sort_idx]
sorted_rho_i  = rho_i_vals[sort_idx]

y_pos = np.arange(len(snames))
bh = 0.35

for j, (lbl, rc, ri) in enumerate(zip(sorted_labels, sorted_rho_c, sorted_rho_i)):
    ax2.barh(j + bh/2, rc, bh, color=rho_color(rc), alpha=0.85, zorder=3)
    ax2.barh(j - bh/2, ri, bh, color=rho_color(ri), alpha=0.55, zorder=3,
             hatch="////" if ri >= 1.0 else None,
             edgecolor=rho_color(ri))

ax2.axvline(1.0, color=C_RED, lw=2.0, ls="--", zorder=5, label="ρ = 1.0  saturation limit")
ax2.axvline(0.8, color=C_AMB, lw=1.3, ls=":",  zorder=5, label="ρ = 0.8  design target")

ax2.set_yticks(y_pos)
ax2.set_yticklabels(sorted_labels, fontsize=7.5)
ax2.set_xlabel("Utilisation factor  ρ = λ · E[S]", fontsize=8.5)
sty(ax2, "2 · Bottleneck Map — Stage Utilisation ρ = λ·E[S]",
    grid_axis="x")
ax2.grid(axis="x", color=C_BORD, lw=0.7, ls="--", zorder=0)
ax2.set_axisbelow(True)

# Legend below chart
ax2.legend(handles=[
    mpatches.Patch(fc=C_RED,  alpha=0.85, label="ρ ≥ 1.0  Saturated (bottleneck)"),
    mpatches.Patch(fc=C_AMB,  alpha=0.85, label="0.8 ≤ ρ < 1.0  High load"),
    mpatches.Patch(fc=C_GRN2, alpha=0.85, label="ρ < 0.8  Within target"),
    plt.Line2D([0],[0], color=C_RED, lw=2, ls="--", label="ρ=1.0 limit"),
    plt.Line2D([0],[0], color=C_AMB, lw=1.3, ls=":", label="ρ=0.8 target"),
    mpatches.Patch(fc="#aaa", alpha=0.85, label="Solid = AS-IS"),
    mpatches.Patch(fc="#aaa", alpha=0.55, hatch="////", ec="#aaa", label="Hatched = TO-BE"),
], fontsize=6.5, loc="center left", ncol=1,
   bbox_to_anchor=(1.01, 0.5),
   facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 3 — M/G/1 Queue Waiting Time Wq (Pollaczek-Khinchine)
# This is the actual queueing analysis: how long does a job WAIT at each stage?
# Saturated stages shown as hatched bars with "∞ Unstable" annotation.
# ═══════════════════════════════════════════════════════════════════════════

# Cap Wq at a display ceiling for plotting; annotate saturated stages explicitly
WQ_CAP = 110.0
wq_c_plot = np.where(np.isfinite(mq_c["Wq"].values), mq_c["Wq"].values, WQ_CAP)
wq_i_plot = np.where(np.isfinite(mq_i["Wq"].values), mq_i["Wq"].values, WQ_CAP)
sat_c = ~np.isfinite(mq_c["Wq"].values)
sat_i = ~np.isfinite(mq_i["Wq"].values)

xs3 = np.arange(len(snames))
bw3 = 0.38

bars_c = ax3.bar(xs3 - bw3/2, wq_c_plot, bw3,
                 color=[C_SAT if s else C_RED for s in sat_c],
                 alpha=0.82, zorder=3,
                 hatch=None)
bars_i = ax3.bar(xs3 + bw3/2, wq_i_plot, bw3,
                 color=[C_SAT if s else C_GRN2 for s in sat_i],
                 alpha=0.82, zorder=3)

# Mark saturated bars
for i, (sc_flag, si_flag) in enumerate(zip(sat_c, sat_i)):
    if sc_flag:
        ax3.text(i - bw3/2, WQ_CAP + 1.5, "∞", ha="center",
                 fontsize=9, color=C_SAT, fontweight="bold")
    if si_flag:
        ax3.text(i + bw3/2, WQ_CAP + 1.5, "∞", ha="center",
                 fontsize=9, color=C_SAT, fontweight="bold")

# Cap line
ax3.axhline(WQ_CAP, color=C_SAT, lw=1.0, ls="--", alpha=0.5, zorder=2)
ax3.text(len(snames)-0.5, WQ_CAP + 1.5, "display cap (∞ above)",
         fontsize=6.5, color=C_SAT, ha="right", style="italic")

ax3.set_xticks(xs3)
ax3.set_xticklabels(slabels, rotation=50, ha="right", fontsize=6.5)
sty(ax3, "3 · M/G/1 Queue Waiting Time  Wq  per Stage",
    yl="Expected waiting time Wq (days)")
ax3.text(0.01, 0.97,
         r"$W_q = \frac{\rho}{1-\rho} \cdot \frac{E[S]}{2} \cdot (1+C_s^2)$",
         transform=ax3.transAxes, fontsize=8, color="#333",
         va="top", bbox=dict(boxstyle="round,pad=0.3", fc=C_LGRY, ec=C_BORD, lw=0.8))

ax3.legend(handles=[
    mpatches.Patch(fc=C_RED,  alpha=0.82, label="AS-IS  Wq (days waiting)"),
    mpatches.Patch(fc=C_GRN2, alpha=0.82, label="TO-BE  Wq (days waiting)"),
    mpatches.Patch(fc=C_SAT,  alpha=0.82, label="Saturated — queue unstable (ρ ≥ 1)"),
], fontsize=7.2, loc="upper left",
   facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 4 — Rework Loop Cost per Decision Gate
# E[wasted days] = p/(1-p) × μ × 0.8  (learning-curve iteration assumption)
# ═══════════════════════════════════════════════════════════════════════════
rw_keys  = list(REWORK_C.keys())
rw_lbls  = [k.replace("\n"," ") for k in rw_keys]
exp_rw_c = [REWORK_C[k]/(1-REWORK_C[k]) * STAGES_C[k]["mu"] * 0.8 for k in rw_keys]
exp_rw_i = [REWORK_I[k]/(1-REWORK_I[k]) * STAGES_I[k]["mu"] * 0.8 for k in rw_keys]

x4 = np.arange(len(rw_keys))
bw4 = 0.32

ax4.bar(x4 - bw4/2, exp_rw_c, bw4, color=C_RED,  alpha=0.82, zorder=3)
ax4.bar(x4 + bw4/2, exp_rw_i, bw4, color=C_GRN2, alpha=0.82, zorder=3)

for i, (vc, vi) in enumerate(zip(exp_rw_c, exp_rw_i)):
    pct = (vc - vi) / vc * 100
    ax4.text(i, max(vc, vi) + 0.15, f"↓{pct:.0f}%",
             ha="center", fontsize=7.5, color=C_GRN, fontweight="bold")
    ax4.text(i - bw4/2, vc/2, f"{vc:.1f}d",
             ha="center", fontsize=7, color="white", fontweight="bold", va="center")
    ax4.text(i + bw4/2, vi/2, f"{vi:.1f}d",
             ha="center", fontsize=7, color="white", fontweight="bold", va="center")
    # Rework probability labels on x-axis
    ax4.text(i - bw4/2, -0.25, f"p={int(REWORK_C[rw_keys[i]]*100)}%",
             ha="center", fontsize=6.5, color=C_RED, style="italic")
    ax4.text(i + bw4/2, -0.25, f"p={int(REWORK_I[rw_keys[i]]*100)}%",
             ha="center", fontsize=6.5, color=C_GRN, style="italic")

ax4.set_xticks(x4)
ax4.set_xticklabels(rw_lbls, fontsize=8.5)
ax4.set_ylim(bottom=-0.7)
sty(ax4, "4 · Rework Loop Cost per Decision Gate",
    yl="Expected wasted days / job")
ax4.text(0.02, 0.97,
         r"$E[\mathrm{rework}] = \frac{p}{1-p} \cdot \mu \cdot 0.8$",
         transform=ax4.transAxes, fontsize=8, color="#333",
         va="top", bbox=dict(boxstyle="round,pad=0.3", fc=C_LGRY, ec=C_BORD, lw=0.8))

ax4.legend(handles=[
    mpatches.Patch(fc=C_RED,  alpha=0.82, label="AS-IS  (rework prob p shown below bar)"),
    mpatches.Patch(fc=C_GRN2, alpha=0.82, label="TO-BE  (reduced rework prob)"),
], fontsize=7.2, loc="lower left",
   facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 5 — Improvement Waterfall
# ═══════════════════════════════════════════════════════════════════════════
stage_savings = {}
for s in snames:
    d = sc[f"t_{s}"].mean() - si[f"t_{s}"].mean()
    if d > 0.5:
        stage_savings[s.replace("\n"," ")] = d

wf_labels = list(stage_savings.keys())
wf_values = list(stage_savings.values())
total_sav = sc["total"].mean() - si["total"].mean()
n_wf      = len(wf_labels)

x_base    = 0
x_stages  = list(range(1, n_wf + 1))
x_total   = n_wf + 1
all_x     = [x_base] + x_stages + [x_total]

running = sc["total"].mean()
bottoms = []
for v in wf_values:
    bottoms.append(running - v)
    running -= v

# Baseline
ax5.bar(x_base, sc["total"].mean(), color=C_RED, alpha=0.85,
        width=0.6, zorder=3, edgecolor="white", lw=1.0)
ax5.text(x_base, sc["total"].mean() + 1.5,
         f"{sc['total'].mean():.0f}d", ha="center",
         fontsize=7.5, color=C_RED, fontweight="bold")
ax5.plot([x_base+0.3, x_stages[0]-0.3],
         [sc["total"].mean(), sc["total"].mean()],
         color=C_GRY, lw=0.8, ls="--", zorder=2)

# Stage bars
for i, (lbl, val, bot) in enumerate(zip(wf_labels, wf_values, bottoms)):
    xi = x_stages[i]
    ax5.bar(xi, val, bottom=bot, color=C_GRN2, alpha=0.85,
            width=0.6, zorder=3, edgecolor="white", lw=1.0)
    ax5.text(xi, bot + val + 0.8, f"−{val:.1f}d",
             ha="center", fontsize=6.5, color=C_GRN, fontweight="bold")
    next_x = x_stages[i+1] if i < n_wf-1 else x_total
    ax5.plot([xi+0.3, next_x-0.3], [bot, bot],
             color=C_GRY, lw=0.8, ls="--", zorder=2)

# TO-BE total
ax5.bar(x_total, si["total"].mean(), color=C_GRN, alpha=0.88,
        width=0.6, zorder=3, edgecolor="white", lw=1.0)
ax5.text(x_total, si["total"].mean() + 1.5,
         f"{si['total'].mean():.0f}d", ha="center",
         fontsize=7.5, color=C_GRN, fontweight="bold")

ax5.set_xticks(all_x)
ax5.set_xticklabels(["AS-IS\nBaseline"] + wf_labels + ["TO-BE\nTotal"],
                    rotation=35, ha="right", fontsize=6.5)
ax5.set_xlim(-0.5, x_total + 0.5)
ax5.set_ylim(si["total"].mean() - 10, sc["total"].mean() + 18)
sty(ax5, "5 · Improvement Waterfall — Where Each Day of Saving Comes From",
    yl="Cumulative cycle time (days)")

ax5.legend(handles=[
    mpatches.Patch(fc=C_RED,  alpha=0.85, label="AS-IS total cycle time"),
    mpatches.Patch(fc=C_GRN2, alpha=0.85, label="Saving from stage improvement"),
    mpatches.Patch(fc=C_GRN,  alpha=0.88, label="TO-BE total cycle time"),
], fontsize=7.2, loc="upper left",
   facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

ax5.text(0.98, 0.04,
         f"Total: −{total_sav:.1f}d  (↓{savings_pct:.0f}%)",
         transform=ax5.transAxes, ha="right", va="bottom",
         fontsize=8.5, fontweight="bold", color=C_GRN,
         bbox=dict(boxstyle="round,pad=0.3", fc="#e8f5e9", ec=C_GRN, lw=1.5))

# ═══════════════════════════════════════════════════════════════════════════
# CHART 6 — Process Variability: Coefficient of Variation per Stage
# CoV = σ/μ — amplifies Wq via Cs² term in P-K formula.
# High CoV stages are inherently unpredictable and drive queue instability.
# ═══════════════════════════════════════════════════════════════════════════
cov_c = mq_c["CoV"].values
cov_i = mq_i["CoV"].values
xs6   = np.arange(len(snames))
bw6   = 0.38

ax6.bar(xs6 - bw6/2, cov_c, bw6, color=C_RED,  alpha=0.82, zorder=3,
        label="AS-IS  CoV = σ/μ")
ax6.bar(xs6 + bw6/2, cov_i, bw6, color=C_GRN2, alpha=0.82, zorder=3,
        label="TO-BE  CoV = σ/μ")

# Reference lines
ax6.axhline(0.5, color=C_AMB, lw=1.5, ls="--", zorder=5,
            label="CoV = 0.5  high variability threshold")
ax6.axhline(0.33, color=C_GRN, lw=1.2, ls=":", zorder=5,
            label="CoV = 0.33  target (truncated-Normal)")

ax6.set_xticks(xs6)
ax6.set_xticklabels(slabels, rotation=50, ha="right", fontsize=6.5)
sty(ax6, "6 · Process Variability — Coefficient of Variation  CoV = σ/μ",
    yl="CoV  (higher = more unpredictable)")
ax6.text(0.01, 0.97,
         "High CoV amplifies queue\nwaiting via Cs² in P-K formula",
         transform=ax6.transAxes, fontsize=7, color=C_GRY,
         va="top", style="italic")

ax6.legend(fontsize=7.2, loc="lower left",
           facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 7 — Sensitivity Tornado: One-at-a-Time Stage Improvement
# Shows which single stage, if improved to TO-BE spec, moves the needle most.
# Ranked by cycle time saving. Full-width bottom row.
# ═══════════════════════════════════════════════════════════════════════════
colors_tornado = [C_GRN if d > 0 else C_GRY for d in df_sens["delta"]]
bars = ax7.barh(range(len(df_sens)), df_sens["delta"],
                color=colors_tornado, alpha=0.85, zorder=3, height=0.6)

# Value labels
for i, (val, lbl) in enumerate(zip(df_sens["delta"], df_sens["label"])):
    if val > 0.3:
        ax7.text(val + 0.15, i, f"−{val:.1f}d", va="center",
                 fontsize=8, color=C_GRN, fontweight="bold")
    elif val <= 0:
        ax7.text(val - 0.15, i, f"{val:.1f}d", va="center",
                 fontsize=8, color=C_GRY, ha="right")

ax7.set_yticks(range(len(df_sens)))
ax7.set_yticklabels(df_sens["label"], fontsize=8.5)
ax7.axvline(0, color="#333", lw=1.0, zorder=4)

sty(ax7,
    "7 · Sensitivity Tornado — One-at-a-Time Stage Improvement "
    "(hold all others at AS-IS, improve one stage to TO-BE spec)",
    xl="Reduction in mean total cycle time (days)", grid_axis="x")
ax7.grid(axis="x", color=C_BORD, lw=0.7, ls="--", zorder=0)
ax7.set_axisbelow(True)

ax7.legend(handles=[
    mpatches.Patch(fc=C_GRN, alpha=0.85,
                   label="Saving in mean cycle time when that stage alone is upgraded to TO-BE"),
], fontsize=8, loc="lower right",
   facecolor=C_WHITE, edgecolor=C_BORD, framealpha=0.95)

# ═══════════════════════════════════════════════════════════════════════════
# CHART 8 — KPI Impact Panel
# ═══════════════════════════════════════════════════════════════════════════
ax8.set_facecolor("#1a1a2e")
ax8.set_xlim(0, 1); ax8.set_ylim(0, 1)
for sp in ax8.spines.values(): sp.set_visible(False)
ax8.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# Compute KPI values from simulation data
mean_c    = sc["total"].mean()
mean_i    = si["total"].mean()
std_c     = sc["total"].std()
std_i     = si["total"].std()
p95_c     = np.percentile(sc["total"], 95)
p95_i     = np.percentile(si["total"], 95)
rw_cols_c = [col for col in sc.columns if col.startswith("rw_")]
rw_cols_i = [col for col in si.columns if col.startswith("rw_")]
avg_rw_c  = sc[rw_cols_c].sum(axis=1).mean()
avg_rw_i  = si[rw_cols_i].sum(axis=1).mean()
sat_count = int((mq_c["rho"] >= 1).sum())

kpis = [
    ("MEAN CYCLE TIME",
     f"{mean_c:.0f}d  →  {mean_i:.0f}d",
     f"↓ {mean_c - mean_i:.0f} days  ({savings_pct:.0f}%)",
     "#e74c3c", "#2ecc71"),
    ("STD DEVIATION",
     f"{std_c:.0f}d  →  {std_i:.0f}d",
     f"↓ {std_c - std_i:.0f} days  ({(std_c-std_i)/std_c*100:.0f}%)",
     "#e67e22", "#f39c12"),
    ("P95 WORST CASE",
     f"{p95_c:.0f}d  →  {p95_i:.0f}d",
     f"↓ {p95_c - p95_i:.0f} days  ({(p95_c-p95_i)/p95_c*100:.0f}%)",
     "#9b59b6", "#8e44ad"),
    ("AVG REWORK LOOPS",
     f"{avg_rw_c:.2f}  →  {avg_rw_i:.2f}  per job",
     f"↓ {(avg_rw_c-avg_rw_i)/avg_rw_c*100:.0f}% rework reduction",
     "#c0392b", "#e74c3c"),
    ("BOTTLENECK STAGES",
     f"{sat_count} saturated  →  1 marginal",
     f"ρ ≥ 1 resolved by automation",
     "#e67e22", "#f39c12"),
    ("QUEUE INSTABILITY",
     "7 stages  Wq → ∞",
     "Eliminated in TO-BE process",
     "#c0392b", "#27ae60"),
]

ax8.text(0.5, 0.975, "8 · PROCESS IMPROVEMENT  KPIs",
         ha="center", va="top", fontsize=8.5, fontweight="bold",
         color="white", transform=ax8.transAxes)

row_h = 0.135
for idx, (title, values, impact, col_as, col_imp) in enumerate(kpis):
    yc = 0.875 - idx * row_h
    # horizontal rule
    ax8.plot([0.03, 0.97], [yc + row_h * 0.55, yc + row_h * 0.55],
             color="#333355", lw=0.6, transform=ax8.transAxes, zorder=1)
    ax8.text(0.5, yc + 0.028, title,
             ha="center", va="center", fontsize=6.5,
             color="#8899aa", fontweight="bold", transform=ax8.transAxes)
    ax8.text(0.5, yc - 0.015, values,
             ha="center", va="center", fontsize=8.5, fontweight="bold",
             color="white", transform=ax8.transAxes)
    ax8.text(0.5, yc - 0.055, impact,
             ha="center", va="center", fontsize=7.2,
             color="#2ecc71", transform=ax8.transAxes)

# ─────────────────────────────────────────────────────────────────────────────
# FOOTER
# ─────────────────────────────────────────────────────────────────────────────
fig.text(0.5, 0.018,
         "M/G/1 P-K formula: Wq = ρ/(1−ρ) · E[S]/2 · (1+Cs²)  where  Cs² = Var[S]/E[S]²  "
         "and  ρ = λ·E[S].   "
         "Rework: geometric loop model, iteration cost = 0.8×μ (learning-curve).   "
         "Sensitivity: n=2,000 per scenario, fixed seed.",
         ha="center", fontsize=7, color=C_GRY, style="italic")

out = "/home/seanhegede/procurement_analytics_v3.png"
fig.savefig(out, dpi=155, bbox_inches="tight", facecolor=C_WHITE)
print(f"Saved → {out}")
print(f"AS-IS: {sc['total'].mean():.1f}d  TO-BE: {si['total'].mean():.1f}d  "
      f"Saving: {savings_pct:.1f}%")
print(f"\nAS-IS saturated stages (ρ≥1):")
for _, r in mq_c[mq_c["rho"]>=1].iterrows():
    print(f"  {r['label']:35s} ρ={r['rho']:.2f}")
print(f"\nTO-BE remaining saturated stages:")
for _, r in mq_i[mq_i["rho"]>=1].iterrows():
    print(f"  {r['label']:35s} ρ={r['rho']:.2f}")

Saved → /home/seanhegede/procurement_analytics_v3.png
AS-IS: 138.8d  TO-BE: 93.6d  Saving: 32.6%

AS-IS saturated stages (ρ≥1):
  Budget Approval                     ρ=1.63
  Shortlist Suppliers                 ρ=1.87
  RFQ                                 ρ=1.68
  Technical Review                    ρ=1.20
  Financial Review                    ρ=1.10
  Negotiations                        ρ=1.81
  Negotiations Finalized              ρ=1.38

TO-BE remaining saturated stages:
  Negotiations Finalized              ρ=1.04
